<a href="https://colab.research.google.com/github/Nefeli-Apostolou/GM-Project---Fraud-Detection/blob/main/Graph_Mining_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---------------------------------------------------------
# 0. Install Dependences
---------------------------------------------------------

In [ ]:
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import networkx as nx
import numpy as np
import torch
import os
from torch_geometric.data import Data
import torch.nn.functional as F
from collections import defaultdict
from tqdm.notebook import tqdm
from torch_geometric.nn import SAGEConv
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

CSV_PATH = "/content/drive/MyDrive/Graph_Mining_Project/eth_tx_last4days.csv"



 ---------------------------------------------------------
# 1. Load transaction data
 ---------------------------------------------------------


In [ ]:

tx = pd.read_csv(CSV_PATH)

required_cols = [
    "block_number",
    "hash",
    "from_address",
    "to_address",
    "value",
    "block_timestamp",
]

missing = set(required_cols) - set(tx.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")



 ---------------------------------------------------------
# 2. Clean addresses
 ---------------------------------------------------------


In [ ]:

tx["from_address"] = tx["from_address"].astype(str).str.lower()
tx["to_address"] = tx["to_address"].astype(str).str.lower()

# Remove contract creation transactions
# (transactions with missing destination address)

tx = tx[
    ~tx["to_address"].isin(["nan", "none", "null", ""])
].copy()

# Also remove actual NaN values if present
tx = tx.dropna(subset=["from_address", "to_address"])

print(f"Remaining transactions after removing contract creations: {len(tx):,}")


Remaining transactions after removing contract creations: 6,092,594



 ---------------------------------------------------------
# 3. Convert value from Wei to ETH
 ---------------------------------------------------------


In [ ]:

tx["value"] = pd.to_numeric(tx["value"], errors="coerce").fillna(0.0)
tx["value_eth"] = tx["value"] / 1e18



 ---------------------------------------------------------
# 4. Convert timestamp
 ---------------------------------------------------------


In [ ]:

tx["block_timestamp"] = pd.to_datetime(tx["block_timestamp"], errors="coerce")

# fallback if timestamp is Unix seconds
if tx["block_timestamp"].isna().mean() > 0.5:
    tx["block_timestamp"] = pd.to_datetime(
        tx["block_timestamp"],
        unit="s",
        errors="coerce"
    )

tx = tx.dropna(subset=["block_timestamp"])

tx["timestamp_unix"] = tx["block_timestamp"].astype("int64") // 10**9



 ---------------------------------------------------------
# 5. Create node mapping
 ---------------------------------------------------------


In [ ]:

all_addresses = pd.concat(
    [tx["from_address"], tx["to_address"]],
    ignore_index=True
).drop_duplicates()

node_id = pd.Series(
    data=np.arange(len(all_addresses), dtype=np.int64),
    index=all_addresses.values
)

tx["src"] = tx["from_address"].map(node_id).astype(np.int64)
tx["dst"] = tx["to_address"].map(node_id).astype(np.int64)

num_nodes = len(node_id)

print(f"Number of nodes: {num_nodes:,}")
print(f"Number of directed edges: {len(tx):,}")


Number of nodes: 1,828,910
Number of directed edges: 6,092,594



 ---------------------------------------------------------
# 6. Build edge_index
 ---------------------------------------------------------


In [ ]:

edge_index = torch.tensor(
    tx[["src", "dst"]].values.T,
    dtype=torch.long
)

# Edge attributes: value_eth and timestamp only
edge_attr_np = tx[["value_eth", "timestamp_unix"]].copy()
edge_attr_np = edge_attr_np.replace([np.inf, -np.inf], np.nan).fillna(0.0)

edge_attr = torch.tensor(
    edge_attr_np.values,
    dtype=torch.float
)



 ---------------------------------------------------------
# 7. Basic flow features
 ---------------------------------------------------------


In [ ]:

out_stats = tx.groupby("src").agg(
    mean_send_amount=("value_eth", "mean"),
    max_send_amount=("value_eth", "max"),
)

in_stats = tx.groupby("dst").agg(
    mean_recv_amount=("value_eth", "mean"),
    max_recv_amount=("value_eth", "max"),
)

# These are used internally to compute ratios, but dropped later
tmp_out = tx.groupby("src").agg(
    send_num=("hash", "count"),
    send_amount=("value_eth", "sum"),
)

tmp_in = tx.groupby("dst").agg(
    recv_num=("hash", "count"),
    recv_amount=("value_eth", "sum"),
)

node_features = pd.DataFrame(index=np.arange(num_nodes))

node_features = node_features.join(out_stats, how="left")
node_features = node_features.join(in_stats, how="left")
node_features = node_features.join(tmp_out, how="left")
node_features = node_features.join(tmp_in, how="left")
node_features = node_features.fillna(0.0)

node_features["total_tx"] = (
    node_features["send_num"] + node_features["recv_num"]
)

node_features["total_amount"] = (
    node_features["send_amount"] + node_features["recv_amount"]
)

node_features["out_ratio"] = (
    node_features["send_num"] /
    node_features["total_tx"].replace(0, np.nan)
).fillna(0.0)

node_features["in_ratio"] = (
    node_features["recv_num"] /
    node_features["total_tx"].replace(0, np.nan)
).fillna(0.0)

node_features["amount_out_ratio"] = (
    node_features["send_amount"] /
    node_features["total_amount"].replace(0, np.nan)
).fillna(0.0)

node_features["amount_in_ratio"] = (
    node_features["recv_amount"] /
    node_features["total_amount"].replace(0, np.nan)
).fillna(0.0)



 ---------------------------------------------------------
# 8. Structural graph features
 ---------------------------------------------------------


In [ ]:

# We aggregate repeated transactions between the same pair of addresses.
# This makes centrality computation much cheaper.

edge_weights = (
    tx.groupby(["src", "dst"])
    .agg(
        tx_count=("hash", "count"),
        total_value=("value_eth", "sum")
    )
    .reset_index()
)

print(f"Aggregated directed edges: {len(edge_weights):,}")

# Directed graph for PageRank and directed degrees
G_dir = nx.DiGraph()
G_dir.add_nodes_from(range(num_nodes))
G_dir.add_weighted_edges_from(
    edge_weights[["src", "dst", "tx_count"]].itertuples(index=False, name=None),
    weight="weight"
)

# Undirected graph for clustering, eigenvector, betweenness
G_undir = nx.Graph()
G_undir.add_nodes_from(range(num_nodes))
G_undir.add_weighted_edges_from(
    edge_weights[["src", "dst", "tx_count"]].itertuples(index=False, name=None),
    weight="weight"
)

# Directed unique-neighbor degrees
in_degree_dict = dict(G_dir.in_degree())
out_degree_dict = dict(G_dir.out_degree())

node_features["in_degree"] = pd.Series(in_degree_dict)
node_features["out_degree"] = pd.Series(out_degree_dict)

# PageRank
print("Computing PageRank...")
pagerank_dict = nx.pagerank(
    G_dir,
    alpha=0.85,
    max_iter=100,
    tol=1e-06,
    weight="weight"
)

node_features["pagerank"] = pd.Series(pagerank_dict)

'''
# Clustering coefficient
print("Computing clustering coefficient...")
clustering_dict = nx.clustering(
    G_undir,
    weight="weight"
)

node_features["clustering_coefficient"] = pd.Series(clustering_dict)


# Eigenvector centrality
# This can be slow on very large graphs.
print("Computing eigenvector centrality...")
try:
    eigen_dict = nx.eigenvector_centrality(
        G_undir,
        max_iter=300,
        tol=1e-06,
        weight="weight"
    )
except nx.PowerIterationFailedConvergence:
    print("Eigenvector centrality did not converge; filling with zeros.")
    eigen_dict = {i: 0.0 for i in range(num_nodes)}

node_features["eigenvector_centrality"] = pd.Series(eigen_dict)
'''

# Approximate betweenness centrality
# Exact betweenness is usually impossible on million-node graphs.
# Increase k for better accuracy, decrease k for speed.
print("Computing approximate betweenness centrality...")
BETWEENNESS_SAMPLE_SIZE = min(1000, num_nodes)

betweenness_dict = nx.betweenness_centrality(
    G_undir,
    k=BETWEENNESS_SAMPLE_SIZE,
    normalized=True,
    weight=None,
    seed=42
)

node_features["betweenness"] = pd.Series(betweenness_dict)

node_features = node_features.fillna(0.0)


Aggregated directed edges: 2,948,955
Computing PageRank...
Computing approximate betweenness centrality...



 ---------------------------------------------------------
# 9. Temporal node features
 ---------------------------------------------------------


In [ ]:

events_out = tx[["src", "timestamp_unix", "block_timestamp"]].copy()
events_out = events_out.rename(columns={"src": "node"})

events_in = tx[["dst", "timestamp_unix", "block_timestamp"]].copy()
events_in = events_in.rename(columns={"dst": "node"})

events = pd.concat([events_out, events_in], ignore_index=True)
events = events.sort_values(["node", "timestamp_unix"])

events["node_inter_tx_time"] = (
    events.groupby("node")["timestamp_unix"].diff()
)

events["node_inter_tx_time"] = (
    events["node_inter_tx_time"]
    .replace([np.inf, -np.inf], np.nan)
)

inter_stats = events.groupby("node")["node_inter_tx_time"].agg(
    mean_inter_tx_time="mean",
    std_inter_tx_time="std"
).fillna(0.0)

node_features = node_features.join(inter_stats, how="left")
node_features = node_features.fillna(0.0)

mu = node_features["mean_inter_tx_time"]
sigma = node_features["std_inter_tx_time"]

node_features["burstiness"] = (
    (sigma - mu) /
    (sigma + mu).replace(0, np.nan)
).fillna(0.0)

time_span_stats = events.groupby("node")["timestamp_unix"].agg(
    first_tx_time="min",
    last_tx_time="max",
    tx_event_count="count"
)

time_span_stats["active_seconds"] = (
    time_span_stats["last_tx_time"] - time_span_stats["first_tx_time"]
)

time_span_stats["active_days"] = (
    time_span_stats["active_seconds"] / 86400
)

active_hours = (time_span_stats["active_seconds"] / 3600).clip(lower=1)

time_span_stats["tx_per_hour"] = (
    time_span_stats["tx_event_count"] / active_hours
)

node_features = node_features.join(
    time_span_stats[
        [
            "active_days",
            "tx_per_hour",
        ]
    ],
    how="left"
)

node_features = node_features.fillna(0.0)

# Night activity: 00:00–06:00 UTC
events["hour"] = events["block_timestamp"].dt.hour
events["is_night"] = events["hour"].between(0, 5).astype(float)

night_stats = events.groupby("node")["is_night"].mean()
night_stats.name = "night_activity_ratio"

node_features = node_features.join(night_stats, how="left")
node_features["night_activity_ratio"] = (
    node_features["night_activity_ratio"].fillna(0.0)
)



 ---------------------------------------------------------
# 10. Remove unwanted/redundant features
 ---------------------------------------------------------


In [ ]:

drop_cols = [
    "send_num",
    "recv_num",
    "send_amount",
    "recv_amount",
]

node_features = node_features.drop(columns=drop_cols, errors="ignore")



 ---------------------------------------------------------
# 11. Clean and log-transform selected features
 ---------------------------------------------------------


In [ ]:

node_features = node_features.replace([np.inf, -np.inf], 0.0)
node_features = node_features.fillna(0.0)

# Do not log-transform ratios, burstiness, PageRank, clustering, eigenvector, betweenness.
log_cols = [
    "mean_send_amount",
    "mean_recv_amount",
    "max_send_amount",
    "max_recv_amount",
    "total_tx",
    "total_amount",
    "in_degree",
    "out_degree",
    "mean_inter_tx_time",
    "std_inter_tx_time",
    "active_days",
    "tx_per_hour",
]

for col in log_cols:
    if col in node_features.columns:
        node_features[col] = np.log1p(node_features[col].clip(lower=0))

x = torch.tensor(
    node_features.values,
    dtype=torch.float
)

print("Node feature matrix shape:", x.shape)
print("Node features:")
print(node_features.columns.tolist())

Node feature matrix shape: torch.Size([1828910, 20])
Node features:
['mean_send_amount', 'max_send_amount', 'mean_recv_amount', 'max_recv_amount', 'total_tx', 'total_amount', 'out_ratio', 'in_ratio', 'amount_out_ratio', 'amount_in_ratio', 'in_degree', 'out_degree', 'pagerank', 'betweenness', 'mean_inter_tx_time', 'std_inter_tx_time', 'burstiness', 'active_days', 'tx_per_hour', 'night_activity_ratio']


 ---------------------------------------------------------
# 12. Create PyTorch Geometric graph object
 ---------------------------------------------------------


In [ ]:

data = Data(
    x=x,
    edge_index=edge_index,
    edge_attr=edge_attr,
    num_nodes=num_nodes
)

print(data)

print("\nNode feature matrix shape:")
print(data.x.shape)

print("\nEdge index shape:")
print(data.edge_index.shape)

print("\nEdge attribute shape:")
print(data.edge_attr.shape)


Data(x=[1828910, 20], edge_index=[2, 6092594], edge_attr=[6092594, 2], num_nodes=1828910)

Node feature matrix shape:
torch.Size([1828910, 20])

Edge index shape:
torch.Size([2, 6092594])

Edge attribute shape:
torch.Size([6092594, 2])



---------------------------------------------------------
# 13. Save outputs
---------------------------------------------------------


In [ ]:

OUTPUT_DIR = "/content/drive/MyDrive/Graph_Mining_Project/ethereum_gnn/"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Main PyTorch Geometric graph
torch.save(
    data,
    os.path.join(OUTPUT_DIR, "ethereum_full_graph.pt")
)

# Address ↔ node ID mapping
node_mapping_df = pd.DataFrame({
    "address": node_id.index,
    "node_id": node_id.values
})

node_mapping_df.to_csv(
    os.path.join(OUTPUT_DIR, "node_mapping.csv"),
    index=False
)

# Human-readable node features
node_features_export = node_features.copy()
node_features_export["node_id"] = node_features_export.index

node_features_export.to_csv(
    os.path.join(OUTPUT_DIR, "node_features.csv"),
    index=False
)

# Aggregated edge list
edge_weights.to_csv(
    os.path.join(OUTPUT_DIR, "aggregated_edges.csv"),
    index=False
)

print("\nSaved files:")
print("- ethereum_full_graph.pt")
print("- node_mapping.csv")
print("- node_features.csv")
print("- aggregated_edges.csv")

# Aggregated edge list:
# Each row represents a unique directed connection between two addresses.
#
# Columns:
# - src: source node ID
# - dst: destination node ID
# - tx_count: number of transactions between the two nodes
# - total_value: total ETH transferred across those transactions
#
# This compressed graph is useful for:
# - temporal motif extraction
# - community detection
# - graph visualization
# - debugging and inspection
# - temporal aggregation
# - classical network analysis
#
# Using aggregated edges is much more memory-efficient than repeatedly
# processing the raw transaction table.

#

 ---------------------------------------------------------
# Part 2.0 Labelling
 ---------------------------------------------------------

In [ ]:

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------

TX_CSV_PATH = "/content/drive/MyDrive/Graph_Mining_Project/eth_tx_last4days.csv"
SCAM_CSV_PATH = "/content/drive/MyDrive/DD_Project/merged_scams.csv"

GRAPH_DIR = "/content/drive/MyDrive/Graph_Mining_Project/ethereum_gnn"
NODE_MAPPING_PATH = os.path.join(GRAPH_DIR, "node_mapping.csv")

# ---------------------------------------------------------
# Helper function
# ---------------------------------------------------------

def normalize_address_col(s):
    return (
        s.astype(str)
        .str.lower()
        .str.strip()
    )

# ---------------------------------------------------------
# 1. Load scam address file
# ---------------------------------------------------------

scams = pd.read_csv(SCAM_CSV_PATH)

if "address" not in scams.columns:
    raise ValueError("The scam CSV must contain a column named 'address'.")

scams["address"] = normalize_address_col(scams["address"])

# Remove duplicates because scam accounts may repeat
unique_scam_addresses = set(scams["address"].dropna())

print("Rows in scam file:", len(scams))
print("Unique scam addresses:", len(unique_scam_addresses))

# ---------------------------------------------------------
# 2. Check how many scam accounts are in eth_tx_last4days_2.csv
# ---------------------------------------------------------

tx = pd.read_csv(
    TX_CSV_PATH,
    usecols=["from_address", "to_address"]
)

tx["from_address"] = normalize_address_col(tx["from_address"])
tx["to_address"] = normalize_address_col(tx["to_address"])

# Unique accounts appearing in raw transaction file
tx_accounts = set(tx["from_address"].dropna()) | set(tx["to_address"].dropna())

scams_in_tx = unique_scam_addresses.intersection(tx_accounts)

print("\nRaw transaction CSV:")
print("Unique accounts in eth_tx_last4days_2.csv:", len(tx_accounts))
print("Scam accounts found in transaction CSV:", len(scams_in_tx))
print(
    "Percentage of scam list found in transaction CSV:",
    round(len(scams_in_tx) / len(unique_scam_addresses) * 100, 4),
    "%"
)

# ---------------------------------------------------------
# 3. Check how many scam accounts are in the graph
# ---------------------------------------------------------

node_mapping = pd.read_csv(NODE_MAPPING_PATH)

node_mapping["address"] = normalize_address_col(node_mapping["address"])

graph_accounts = set(node_mapping["address"].dropna())

scams_in_graph = unique_scam_addresses.intersection(graph_accounts)

print("\nCreated graph:")
print("Unique accounts in graph:", len(graph_accounts))
print("Scam accounts found in graph:", len(scams_in_graph))
print(
    "Percentage of scam list found in graph:",
    round(len(scams_in_graph) / len(unique_scam_addresses) * 100, 4),
    "%"
)


# ---------------------------------------------------------
# 5. Find all nodes that transacted with scam accounts
# ---------------------------------------------------------

# Transactions where either side is a known scam
tx_with_scams = tx[
    tx["from_address"].isin(unique_scam_addresses) |
    tx["to_address"].isin(unique_scam_addresses)
].copy()

# Collect counterparties
counterparties_from = set(
    tx_with_scams.loc[
        tx_with_scams["from_address"].isin(unique_scam_addresses),
        "to_address"
    ].dropna()
)

counterparties_to = set(
    tx_with_scams.loc[
        tx_with_scams["to_address"].isin(unique_scam_addresses),
        "from_address"
    ].dropna()
)

# Union of all counterparties
counterparty_nodes = counterparties_from | counterparties_to

# Remove scam accounts themselves
counterparty_nodes = counterparty_nodes - unique_scam_addresses

print("\nCounterparty Analysis")
print("Unique nodes that interacted with scam accounts:",
      len(counterparty_nodes))

# ---------------------------------------------------------
# 6. Check how many of these counterparties are in graph
# ---------------------------------------------------------

counterparties_in_graph = counterparty_nodes.intersection(graph_accounts)

print("\nCounterparties present in graph:")
print(len(counterparties_in_graph))





Rows in scam file: 8430
Unique scam addresses: 6717

Raw transaction CSV:
Unique accounts in eth_tx_last4days_2.csv: 1829064
Scam accounts found in transaction CSV: 131
Percentage of scam list found in transaction CSV: 1.9503 %

Created graph:
Unique accounts in graph: 1746059
Scam accounts found in graph: 117
Percentage of scam list found in graph: 1.7418 %

Counterparty Analysis
Unique nodes that interacted with scam accounts: 679892

Counterparties present in graph:
149977


In [ ]:
# ---------------------------------------------------------
# 4. Save matched scam accounts
# ---------------------------------------------------------

scams_in_tx_df = pd.DataFrame({
    "address": sorted(scams_in_tx)
})

scams_in_graph_df = node_mapping[
    node_mapping["address"].isin(scams_in_graph)
].copy()

scams_in_tx_df.to_csv(
    os.path.join(GRAPH_DIR, "scam_accounts_in_raw_transactions.csv"),
    index=False
)

scams_in_graph_df.to_csv(
    os.path.join(GRAPH_DIR, "scam_accounts_in_graph.csv"),
    index=False
)

print("\nSaved:")
print("- scam_accounts_in_raw_transactions.csv")
print("- scam_accounts_in_graph.csv")


Saved:
- scam_accounts_in_raw_transactions.csv
- scam_accounts_in_graph.csv


In [ ]:
# ---------------------------------------------------------
# Merge known scam accounts with your detected fraud accounts
# ---------------------------------------------------------

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------

GRAPH_DIR = "/content/drive/MyDrive/Graph_Mining_Project/ethereum_gnn/"

# Scam accounts found in your transaction dataset
SCAM_OVERLAP_PATH = os.path.join(
    GRAPH_DIR,
    "scam_accounts_in_raw_transactions.csv"
)

# Your internally detected fraud accounts
DETECTED_FRAUD_PATH = "/content/drive/MyDrive/Graph_Mining_Project/all_detected_fraud_accounts.csv"

# ---------------------------------------------------------
# Load files
# ---------------------------------------------------------

scam_overlap = pd.read_csv(SCAM_OVERLAP_PATH)
detected_fraud = pd.read_csv(DETECTED_FRAUD_PATH)

# ---------------------------------------------------------
# Normalize addresses
# ---------------------------------------------------------

def normalize_address_col(s):
    return (
        s.astype(str)
        .str.lower()
        .str.strip()
    )

# Rename address column if needed
possible_cols = ["address", "wallet", "wallet_address"]

found_col = None
for col in possible_cols:
    if col in detected_fraud.columns:
        found_col = col
        break

if found_col is None:
    raise ValueError(
        f"Could not find an address column in detected fraud file. "
        f"Columns are: {detected_fraud.columns.tolist()}"
    )

detected_fraud = detected_fraud.rename(
    columns={found_col: "address"}
)

# Normalize
scam_overlap["address"] = normalize_address_col(
    scam_overlap["address"]
)

detected_fraud["address"] = normalize_address_col(
    detected_fraud["address"]
)

# ---------------------------------------------------------
# Add source labels
# ---------------------------------------------------------

scam_overlap["source"] = "online_scam_list"
detected_fraud["source"] = "our_detected_fraud"

# ---------------------------------------------------------
# Merge and deduplicate
# ---------------------------------------------------------

merged_fraud = pd.concat(
    [
        scam_overlap[["address", "source"]],
        detected_fraud[["address", "source"]]
    ],
    ignore_index=True
)

# Keep track of addresses appearing in both sources
merged_fraud = (
    merged_fraud
    .groupby("address")["source"]
    .apply(lambda x: ",".join(sorted(set(x))))
    .reset_index()
)

# ---------------------------------------------------------
# Statistics
# ---------------------------------------------------------

print("Online scam accounts in transactions:",
      len(scam_overlap))

print("Our detected fraud accounts:",
      len(detected_fraud))

print("Unique merged fraud accounts:",
      len(merged_fraud))

# Accounts appearing in both lists
both_sources = merged_fraud[
    merged_fraud["source"].str.contains(",")
]

print("Accounts appearing in BOTH sources:",
      len(both_sources))

# ---------------------------------------------------------
# Save merged fraud list
# ---------------------------------------------------------

OUTPUT_PATH = os.path.join(
    GRAPH_DIR,
    "merged_fraud_accounts.csv"
)

merged_fraud.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\nSaved:")
print("- merged_fraud_accounts.csv")

Online scam accounts in transactions: 131
Our detected fraud accounts: 4351
Unique merged fraud accounts: 4466
Accounts appearing in BOTH sources: 16

Saved:
- merged_fraud_accounts.csv


In [ ]:

OUTPUT_DIR = "/content/drive/MyDrive/Graph_Mining_Project/ethereum_gnn"

NODE_MAPPING_PATH = os.path.join(OUTPUT_DIR, "node_mapping.csv")
AGG_EDGES_PATH = os.path.join(OUTPUT_DIR, "aggregated_edges.csv")

# Correct fraud file
FRAUD_PATH = os.path.join(OUTPUT_DIR, "merged_fraud_accounts.csv")

# Infomap communities file
INFOMAP_PATH = "/content/drive/MyDrive/Graph_Mining_Project/infomap_communities.csv"

TARGET_COMMUNITY_NORMALS = 25000
TARGET_RANDOM_NORMALS = 20000
RANDOM_SEED = 42

In [ ]:
# ---------------------------------------------------------
# 1. Load files
# ---------------------------------------------------------

node_mapping = pd.read_csv(NODE_MAPPING_PATH)
edges = pd.read_csv(AGG_EDGES_PATH)
fraud_df = pd.read_csv(FRAUD_PATH)
infomap_df = pd.read_csv(INFOMAP_PATH)

node_mapping["address"] = node_mapping["address"].astype(str).str.lower().str.strip()
infomap_df["address"] = infomap_df["address"].astype(str).str.lower().str.strip()

if "address" in fraud_df.columns:
    fraud_df["address"] = fraud_df["address"].astype(str).str.lower().str.strip()

node_mapping["node_id"] = node_mapping["node_id"].astype(int)
edges["src"] = edges["src"].astype(int)
edges["dst"] = edges["dst"].astype(int)

print("Nodes in graph:", len(node_mapping))
print("Aggregated edges:", len(edges))
print("Merged fraud accounts:", len(fraud_df))
print("Infomap rows:", len(infomap_df))

Nodes in graph: 1828910
Aggregated edges: 2948955
Merged fraud accounts: 4466
Infomap rows: 1686534


In [ ]:
# ---------------------------------------------------------
# 2. Get fraud node IDs from merged_fraud_accounts.csv
# ---------------------------------------------------------

if "node_id" in fraud_df.columns:
    fraud_df["node_id"] = pd.to_numeric(fraud_df["node_id"], errors="coerce")
    fraud_df = fraud_df.dropna(subset=["node_id"])
    fraud_df["node_id"] = fraud_df["node_id"].astype(int)

else:
    fraud_df = fraud_df.merge(
        node_mapping[["address", "node_id"]],
        on="address",
        how="inner"
    )

fraud_nodes = set(fraud_df["node_id"].astype(int))

print("Fraud nodes found in graph:", len(fraud_nodes))

Fraud nodes found in graph: 4466


In [ ]:
# ---------------------------------------------------------
# 3. Attach node_id to Infomap communities
# ---------------------------------------------------------

infomap_df = infomap_df.merge(
    node_mapping[["address", "node_id"]],
    on="address",
    how="inner"
)

infomap_df["node_id"] = infomap_df["node_id"].astype(int)

print("Infomap nodes found in graph:", len(infomap_df))

Infomap nodes found in graph: 1686380


In [ ]:
# ---------------------------------------------------------
# 4. Find Infomap communities containing fraud nodes
# ---------------------------------------------------------

fraud_communities = set(
    infomap_df.loc[
        infomap_df["node_id"].isin(fraud_nodes),
        "infomap_id"
    ]
)

nodes_in_fraud_communities = set(
    infomap_df.loc[
        infomap_df["infomap_id"].isin(fraud_communities),
        "node_id"
    ]
)

print("Fraud-related Infomap communities:", len(fraud_communities))
print("Nodes in fraud-related Infomap communities:", len(nodes_in_fraud_communities))

Fraud-related Infomap communities: 2586
Nodes in fraud-related Infomap communities: 1125121


In [ ]:
# ---------------------------------------------------------
# 5. Find direct neighbors of fraudulent nodes
# ---------------------------------------------------------

fraud_out_edges = edges[edges["src"].isin(fraud_nodes)]
fraud_in_edges = edges[edges["dst"].isin(fraud_nodes)]

direct_neighbors_of_fraud = set(fraud_out_edges["dst"]).union(
    set(fraud_in_edges["src"])
)

print("Direct neighbors of fraud nodes:", len(direct_neighbors_of_fraud))

Direct neighbors of fraud nodes: 1113021


In [ ]:
# ---------------------------------------------------------
# 6. Select 25,000 normal nodes from fraud-related Infomap communities
# ---------------------------------------------------------

community_normal_candidates = (
    nodes_in_fraud_communities
    - fraud_nodes
    - direct_neighbors_of_fraud
)

community_normal_candidates = list(community_normal_candidates)

print("Community normal candidates:", len(community_normal_candidates))

if len(community_normal_candidates) < TARGET_COMMUNITY_NORMALS:
    raise ValueError(
        f"Only {len(community_normal_candidates)} community normal candidates found. "
        f"Need {TARGET_COMMUNITY_NORMALS}."
    )

rng = np.random.default_rng(RANDOM_SEED)

community_normal_nodes = set(
    rng.choice(
        community_normal_candidates,
        size=TARGET_COMMUNITY_NORMALS,
        replace=False
    )
)

print("Selected community normals:", len(community_normal_nodes))

Community normal candidates: 198271
Selected community normals: 25000


In [ ]:
# ---------------------------------------------------------
# 7. Select 20,000 random normal nodes from the rest of the graph
# ---------------------------------------------------------

all_graph_nodes = set(node_mapping["node_id"].astype(int))

random_normal_candidates = (
    all_graph_nodes
    - fraud_nodes
    - direct_neighbors_of_fraud
    - community_normal_nodes
    - nodes_in_fraud_communities
)

random_normal_candidates = list(random_normal_candidates)

print("Random normal candidates from rest of graph:", len(random_normal_candidates))

if len(random_normal_candidates) < TARGET_RANDOM_NORMALS:
    raise ValueError(
        f"Only {len(random_normal_candidates)} random normal candidates found. "
        f"Need {TARGET_RANDOM_NORMALS}."
    )

random_normal_nodes = set(
    rng.choice(
        random_normal_candidates,
        size=TARGET_RANDOM_NORMALS,
        replace=False
    )
)

print("Selected random normals:", len(random_normal_nodes))

Random normal candidates from rest of graph: 516021
Selected random normals: 20000


In [ ]:
# ---------------------------------------------------------
# 8. Create final GNN label table
# ---------------------------------------------------------

fraud_labels = pd.DataFrame({
    "node_id": list(fraud_nodes),
    "label": 1,
    "label_type": "fraud_merged"
})

community_normal_labels = pd.DataFrame({
    "node_id": list(community_normal_nodes),
    "label": 0,
    "label_type": "normal_fraud_infomap_community_not_direct_neighbor"
})

random_normal_labels = pd.DataFrame({
    "node_id": list(random_normal_nodes),
    "label": 0,
    "label_type": "normal_random_rest_of_graph"
})

gnn_labels = pd.concat(
    [fraud_labels, community_normal_labels, random_normal_labels],
    ignore_index=True
)

gnn_labels = gnn_labels.merge(
    node_mapping[["node_id", "address"]],
    on="node_id",
    how="left"
)

gnn_labels = gnn_labels[["node_id", "address", "label", "label_type"]]

print(gnn_labels["label"].value_counts())
print(gnn_labels["label_type"].value_counts())
print("Total labelled nodes:", len(gnn_labels))

label
0    45000
1     4466
Name: count, dtype: int64
label_type
normal_fraud_infomap_community_not_direct_neighbor    25000
normal_random_rest_of_graph                           20000
fraud_merged                                           4466
Name: count, dtype: int64
Total labelled nodes: 49466


In [ ]:
# ---------------------------------------------------------
# 9. Save final label file
# ---------------------------------------------------------

LABEL_OUTPUT_PATH = os.path.join(OUTPUT_DIR, "gnn_labels.csv")

gnn_labels.to_csv(LABEL_OUTPUT_PATH, index=False)

print("Saved:", LABEL_OUTPUT_PATH)

Saved: /content/drive/MyDrive/Graph_Mining_Project/ethereum_gnn/gnn_labels.csv


 =========================================================
# GraphSAGE semi-supervised node classification #1
 =========================================================

## This is our first attempt

In [ ]:

# ---------------------------------------------------------
# 1. Paths
# ---------------------------------------------------------
OUTPUT_DIR = "/content/drive/MyDrive/Graph_Mining_Project/ethereum_gnn/"
GRAPH_PATH = os.path.join(OUTPUT_DIR, "ethereum_full_graph.pt")
LABEL_PATH = os.path.join(OUTPUT_DIR, "gnn_labels.csv")
NODE_MAPPING_PATH = os.path.join(OUTPUT_DIR, "node_mapping.csv")


In [ ]:

# ---------------------------------------------------------
# 2. Load graph and labels
# ---------------------------------------------------------

data = torch.load(
    GRAPH_PATH,
    map_location="cpu",
    weights_only=False
)

labels_df = pd.read_csv(LABEL_PATH)
node_mapping = pd.read_csv(NODE_MAPPING_PATH)

labels_df["node_id"] = labels_df["node_id"].astype(int)
labels_df["label"] = labels_df["label"].astype(int)

print(data)
print(labels_df["label"].value_counts())
print("Total labelled nodes:", len(labels_df))


KeyboardInterrupt: 

In [ ]:

# ---------------------------------------------------------
# 3. Create y vector
# ---------------------------------------------------------
# -1 = unlabeled
#  0 = normal
#  1 = fraud

num_nodes = data.num_nodes

y = torch.full(
    (num_nodes,),
    -1,
    dtype=torch.long
)

labelled_node_ids = labels_df["node_id"].values
label_values = labels_df["label"].values

y[labelled_node_ids] = torch.tensor(
    label_values,
    dtype=torch.long
)

data.y = y

print("Unlabelled nodes:", (data.y == -1).sum().item())
print("Normal labelled nodes:", (data.y == 0).sum().item())
print("Fraud labelled nodes:", (data.y == 1).sum().item())


Unlabelled nodes: 1779444
Normal labelled nodes: 45000
Fraud labelled nodes: 4466


In [ ]:

# ---------------------------------------------------------
# 4. Train / validation / test split
# ---------------------------------------------------------

train_ids, temp_ids = train_test_split(
    labelled_node_ids,
    test_size=0.30,
    random_state=42,
    stratify=label_values
)

temp_labels = y[temp_ids].numpy()

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)

train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)

train_mask[train_ids] = True
val_mask[val_ids] = True
test_mask[test_ids] = True

data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

print("Train nodes:", data.train_mask.sum().item())
print("Validation nodes:", data.val_mask.sum().item())
print("Test nodes:", data.test_mask.sum().item())


Train nodes: 34626
Validation nodes: 7420
Test nodes: 7420


In [ ]:

# ---------------------------------------------------------
# 5. Define GraphSAGE model
# ---------------------------------------------------------

class GraphSAGE(torch.nn.Module):
    def __init__(
        self,
        in_channels,
        hidden_channels,
        out_channels,
        dropout=0.3
    ):
        super().__init__()

        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)

        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        x = self.conv2(x, edge_index)

        return x


In [ ]:

# ---------------------------------------------------------
# 6. Device, model, optimizer
# ---------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

data = data.to(device)

model = GraphSAGE(
    in_channels=data.x.shape[1],
    hidden_channels=64,
    out_channels=2,
    dropout=0.3
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=5e-4
)


Using device: cuda


In [ ]:

# ---------------------------------------------------------
# 7. Class weights for imbalance
# ---------------------------------------------------------

train_labels = data.y[data.train_mask]

num_normal = (train_labels == 0).sum().item()
num_fraud = (train_labels == 1).sum().item()

class_weights = torch.tensor(
    [
        1.0 / num_normal,
        1.0 / num_fraud
    ],
    dtype=torch.float,
    device=device
)

class_weights = class_weights / class_weights.sum() * 2

print("Class weights:", class_weights)


Class weights: tensor([0.1806, 1.8194], device='cuda:0')


In [ ]:

# ---------------------------------------------------------
# 8. Training and evaluation functions
# ---------------------------------------------------------

def train_one_epoch():
    model.train()
    optimizer.zero_grad()

    out = model(data.x, data.edge_index)

    loss = F.cross_entropy(
        out[data.train_mask],
        data.y[data.train_mask],
        weight=class_weights
    )

    loss.backward()
    optimizer.step()

    return loss.item()


@torch.no_grad()
def evaluate(mask):
    model.eval()

    out = model(data.x, data.edge_index)

    logits = out[mask]
    labels = data.y[mask]

    probs = F.softmax(logits, dim=1)[:, 1]
    preds = logits.argmax(dim=1)

    acc = (preds == labels).float().mean().item()

    probs_np = probs.detach().cpu().numpy()
    preds_np = preds.detach().cpu().numpy()
    labels_np = labels.detach().cpu().numpy()

    roc_auc = roc_auc_score(labels_np, probs_np)
    pr_auc = average_precision_score(labels_np, probs_np)

    return acc, roc_auc, pr_auc, preds_np, probs_np, labels_np


In [ ]:

# ---------------------------------------------------------
# 9. Train model
# ---------------------------------------------------------

EPOCHS = 100

best_val_pr_auc = 0
best_model_path = os.path.join(OUTPUT_DIR, "best_graphsage_model.pt")

for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch()

    if epoch % 10 == 0:
        train_acc, train_roc, train_pr, _, _, _ = evaluate(data.train_mask)
        val_acc, val_roc, val_pr, _, _, _ = evaluate(data.val_mask)

        print(
            f"Epoch {epoch:03d} | "
            f"Loss: {loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Train PR-AUC: {train_pr:.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"Val ROC-AUC: {val_roc:.4f} | "
            f"Val PR-AUC: {val_pr:.4f}"
        )

        if val_pr > best_val_pr_auc:
            best_val_pr_auc = val_pr

            torch.save(
                model.state_dict(),
                best_model_path
            )

            print("Saved best model.")


Epoch 010 | Loss: 0.6857 | Train Acc: 0.5006 | Train PR-AUC: 0.3336 | Val Acc: 0.5023 | Val ROC-AUC: 0.8456 | Val PR-AUC: 0.3254
Saved best model.
Epoch 020 | Loss: 0.6035 | Train Acc: 0.6203 | Train PR-AUC: 0.5489 | Val Acc: 0.6275 | Val ROC-AUC: 0.9368 | Val PR-AUC: 0.5402
Saved best model.
Epoch 030 | Loss: 0.4902 | Train Acc: 0.8172 | Train PR-AUC: 0.6231 | Val Acc: 0.8168 | Val ROC-AUC: 0.9253 | Val PR-AUC: 0.6134
Saved best model.
Epoch 040 | Loss: 0.4179 | Train Acc: 0.8456 | Train PR-AUC: 0.7491 | Val Acc: 0.8398 | Val ROC-AUC: 0.9610 | Val PR-AUC: 0.7414
Saved best model.
Epoch 050 | Loss: 0.3634 | Train Acc: 0.8717 | Train PR-AUC: 0.8185 | Val Acc: 0.8682 | Val ROC-AUC: 0.9719 | Val PR-AUC: 0.8107
Saved best model.
Epoch 060 | Loss: 0.3246 | Train Acc: 0.9016 | Train PR-AUC: 0.8479 | Val Acc: 0.8968 | Val ROC-AUC: 0.9747 | Val PR-AUC: 0.8392
Saved best model.
Epoch 070 | Loss: 0.2890 | Train Acc: 0.9062 | Train PR-AUC: 0.8662 | Val Acc: 0.9028 | Val ROC-AUC: 0.9777 | Val PR-A

In [ ]:

# ---------------------------------------------------------
# 10. Final test evaluation
# ---------------------------------------------------------

model.load_state_dict(
    torch.load(
        best_model_path,
        map_location=device
    )
)

test_acc, test_roc, test_pr, test_preds, test_probs, test_labels = evaluate(
    data.test_mask
)

print("\nFinal test results")
print("Test accuracy:", test_acc)
print("Test ROC-AUC:", test_roc)
print("Test PR-AUC:", test_pr)

print("\nClassification report:")
print(
    classification_report(
        test_labels,
        test_preds,
        target_names=["normal", "fraud"]
    )
)

print("\nConfusion matrix:")
print(confusion_matrix(test_labels, test_preds))





Final test results
Test accuracy: 0.9176549911499023
Test ROC-AUC: 0.9841834162520728
Test PR-AUC: 0.8745253229617966

Classification report:
              precision    recall  f1-score   support

      normal       1.00      0.91      0.95      6750
       fraud       0.52      0.98      0.68       670

    accuracy                           0.92      7420
   macro avg       0.76      0.95      0.82      7420
weighted avg       0.95      0.92      0.93      7420


Confusion matrix:
[[6153  597]
 [  14  656]]


In [ ]:
# ---------------------------------------------------------
# 11. Predict fraud probability for every node
# ---------------------------------------------------------
model.eval()

with torch.no_grad():
    out = model(data.x, data.edge_index)
    probs = F.softmax(out, dim=1)
    fraud_probs = probs[:, 1].detach().cpu().numpy()

pred_df = pd.DataFrame({
    "node_id": np.arange(data.num_nodes),
    "fraud_probability": fraud_probs
})

node_mapping["node_id"] = node_mapping["node_id"].astype(int)

pred_df = pred_df.merge(
    node_mapping,
    on="node_id",
    how="left"
)

# Mark whether the node was labelled or unlabeled
y_cpu = data.y.detach().cpu().numpy()

pred_df["known_label"] = y_cpu
pred_df["is_labelled"] = pred_df["known_label"] != -1

pred_df = pred_df.sort_values(
    "fraud_probability",
    ascending=False
)

PRED_OUTPUT_PATH = os.path.join(
    OUTPUT_DIR,
    "graphsage_all_node_predictions.csv"
)

pred_df.to_csv(
    PRED_OUTPUT_PATH,
    index=False
)

print("\nSaved predictions:")
print(PRED_OUTPUT_PATH)

display(pred_df.head(20))


Saved predictions:
/content/drive/MyDrive/Graph_Mining_Project/ethereum_gnn/graphsage_all_node_predictions.csv


,node_id,fraud_probability,address,known_label,is_labelled
4478,4478,0.999765,0xad6eaa735d9df3d7696fd03984379dae02ed8862,1,True
5605,5605,0.999745,0xb5d85cbf7cb3ee0d56b3bb207d5fc4b82f43f511,1,True
29562,29562,0.999733,0xcffad3200574698b78f32232aa9d63eabd290703,1,True
276,276,0.999692,0x091d1c972cb1648537a2ba78eaba371b1ce18336,1,True
169853,169853,0.999663,0x113a1c1294deb0cae51b17a3fdd4c2cf7ba935df,1,True
1241192,1241192,0.999657,0xd0a3f823b16450482e34f1217ddb24817c60cc0c,1,True
7227,7227,0.999592,0xcd531ae9efcce479654c4926dec5f6209531ca7b,1,True
268178,268178,0.999578,0x17e5545b11b468072283cee1f066a059fb0dbf24,1,True
35260,35260,0.999570,0xc7bf35c9a3bdd1b1c19a6963de669cb45191a019,1,True
170290,170290,0.999550,0x98adef6f2ac8572ec48965509d69a8dd5e8bba9d,1,True


 =========================================================
# GraphSAGE semi-supervised node classification #2
 =========================================================

## the model currently uses argmax, which is equivalent to a fixed threshold around 0.5. Since our model has very high recall but low precision, we should choose a stricter fraud threshold using the validation set, then evaluate that chosen threshold once on the test set.

In [ ]:
# =========================================================
# GraphSAGE semi-supervised node classification
# with validation-selected fraud threshold
# =========================================================

# ---------------------------------------------------------
# 1. Paths
# ---------------------------------------------------------

OUTPUT_DIR = "/content/drive/MyDrive/Graph_Mining_Project/ethereum_gnn/"
GRAPH_PATH = os.path.join(OUTPUT_DIR, "ethereum_full_graph.pt")
LABEL_PATH = os.path.join(OUTPUT_DIR, "gnn_labels.csv")
NODE_MAPPING_PATH = os.path.join(OUTPUT_DIR, "node_mapping.csv")

# ---------------------------------------------------------
# 2. Load graph and labels
# ---------------------------------------------------------

data = torch.load(
    GRAPH_PATH,
    map_location="cpu",
    weights_only=False
)

labels_df = pd.read_csv(LABEL_PATH)
node_mapping = pd.read_csv(NODE_MAPPING_PATH)

labels_df["node_id"] = labels_df["node_id"].astype(int)
labels_df["label"] = labels_df["label"].astype(int)

print(data)
print(labels_df["label"].value_counts())
print("Total labelled nodes:", len(labels_df))

# ---------------------------------------------------------
# 3. Create y vector
# ---------------------------------------------------------
# -1 = unlabeled
#  0 = normal
#  1 = fraud

num_nodes = data.num_nodes

y = torch.full(
    (num_nodes,),
    -1,
    dtype=torch.long
)

labelled_node_ids = labels_df["node_id"].values
label_values = labels_df["label"].values

y[labelled_node_ids] = torch.tensor(
    label_values,
    dtype=torch.long
)

data.y = y

print("Unlabelled nodes:", (data.y == -1).sum().item())
print("Normal labelled nodes:", (data.y == 0).sum().item())
print("Fraud labelled nodes:", (data.y == 1).sum().item())

# ---------------------------------------------------------
# 4. Train / validation / test split
# ---------------------------------------------------------

train_ids, temp_ids = train_test_split(
    labelled_node_ids,
    test_size=0.30,
    random_state=42,
    stratify=label_values
)

temp_labels = y[temp_ids].numpy()

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)

train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)

train_mask[train_ids] = True
val_mask[val_ids] = True
test_mask[test_ids] = True

data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

print("Train nodes:", data.train_mask.sum().item())
print("Validation nodes:", data.val_mask.sum().item())
print("Test nodes:", data.test_mask.sum().item())

# ---------------------------------------------------------
# 5. Define GraphSAGE model
# ---------------------------------------------------------

class GraphSAGE(torch.nn.Module):
    def __init__(
        self,
        in_channels,
        hidden_channels,
        out_channels,
        dropout=0.3
    ):
        super().__init__()

        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )
        x = self.conv2(x, edge_index)
        return x

# ---------------------------------------------------------
# 6. Device, model, optimizer
# ---------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

data = data.to(device)

model = GraphSAGE(
    in_channels=data.x.shape[1],
    hidden_channels=64,
    out_channels=2,
    dropout=0.3
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=5e-4
)

# ---------------------------------------------------------
# 7. Class weights for imbalance
# ---------------------------------------------------------

train_labels = data.y[data.train_mask]

num_normal = (train_labels == 0).sum().item()
num_fraud = (train_labels == 1).sum().item()

class_weights = torch.tensor(
    [
        1.0 / num_normal,
        1.0 / num_fraud
    ],
    dtype=torch.float,
    device=device
)

class_weights = class_weights / class_weights.sum() * 2

print("Class weights:", class_weights)

# ---------------------------------------------------------
# 8. Training and evaluation functions
# ---------------------------------------------------------

def train_one_epoch():
    model.train()
    optimizer.zero_grad()

    out = model(data.x, data.edge_index)

    loss = F.cross_entropy(
        out[data.train_mask],
        data.y[data.train_mask],
        weight=class_weights
    )

    loss.backward()
    optimizer.step()

    return loss.item()


@torch.no_grad()
def evaluate(mask):
    model.eval()

    out = model(data.x, data.edge_index)

    logits = out[mask]
    labels = data.y[mask]

    probs = F.softmax(logits, dim=1)[:, 1]
    preds_argmax = logits.argmax(dim=1)

    acc = (preds_argmax == labels).float().mean().item()

    probs_np = probs.detach().cpu().numpy()
    preds_np = preds_argmax.detach().cpu().numpy()
    labels_np = labels.detach().cpu().numpy()

    roc_auc = roc_auc_score(labels_np, probs_np)
    pr_auc = average_precision_score(labels_np, probs_np)

    return acc, roc_auc, pr_auc, preds_np, probs_np, labels_np

# ---------------------------------------------------------
# 9. Train model
# ---------------------------------------------------------

EPOCHS = 100

best_val_pr_auc = 0
best_model_path = os.path.join(OUTPUT_DIR, "best_graphsage_model.pt")

for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch()

    if epoch % 10 == 0:
        train_acc, train_roc, train_pr, _, _, _ = evaluate(data.train_mask)
        val_acc, val_roc, val_pr, _, _, _ = evaluate(data.val_mask)

        print(
            f"Epoch {epoch:03d} | "
            f"Loss: {loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Train PR-AUC: {train_pr:.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"Val ROC-AUC: {val_roc:.4f} | "
            f"Val PR-AUC: {val_pr:.4f}"
        )

        if val_pr > best_val_pr_auc:
            best_val_pr_auc = val_pr

            torch.save(
                model.state_dict(),
                best_model_path
            )

            print("Saved best model.")

# ---------------------------------------------------------
# 10. Load best model
# ---------------------------------------------------------

model.load_state_dict(
    torch.load(
        best_model_path,
        map_location=device
    )
)

# ---------------------------------------------------------
# 11. Select best fraud threshold on validation set
# ---------------------------------------------------------
# Choose the threshold that maximizes F1-score on validation data.

val_acc, val_roc, val_pr, val_preds_argmax, val_probs, val_labels = evaluate(
    data.val_mask
)

thresholds = np.arange(0.05, 0.96, 0.01)

threshold_results = []

for threshold in thresholds:
    val_preds_thresholded = (val_probs >= threshold).astype(int)

    precision = precision_score(
        val_labels,
        val_preds_thresholded,
        zero_division=0
    )

    recall = recall_score(
        val_labels,
        val_preds_thresholded,
        zero_division=0
    )

    f1 = f1_score(
        val_labels,
        val_preds_thresholded,
        zero_division=0
    )

    threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_results = pd.DataFrame(threshold_results)

best_row = threshold_results.loc[
    threshold_results["f1"].idxmax()
]

best_threshold = float(best_row["threshold"])

print("\nBest threshold selected on validation set:")
print(best_row)

# Save threshold search results
threshold_results_path = os.path.join(
    OUTPUT_DIR,
    "graphsage_threshold_results_validation.csv"
)

threshold_results.to_csv(
    threshold_results_path,
    index=False
)

print("Saved threshold results:", threshold_results_path)

# ---------------------------------------------------------
# 12. Final test evaluation with selected threshold
# ---------------------------------------------------------

test_acc_argmax, test_roc, test_pr, test_preds_argmax, test_probs, test_labels = evaluate(
    data.test_mask
)

test_preds_thresholded = (test_probs >= best_threshold).astype(int)

print("\nFinal test results with validation-selected threshold")
print("Selected threshold:", best_threshold)
print("Test ROC-AUC:", test_roc)
print("Test PR-AUC:", test_pr)

print("\nClassification report:")
print(
    classification_report(
        test_labels,
        test_preds_thresholded,
        target_names=["normal", "fraud"],
        zero_division=0
    )
)

print("\nConfusion matrix:")
print(confusion_matrix(test_labels, test_preds_thresholded))

# ---------------------------------------------------------
# 13. Predict fraud probability for every node
# ---------------------------------------------------------

model.eval()

with torch.no_grad():
    out = model(data.x, data.edge_index)
    probs = F.softmax(out, dim=1)
    fraud_probs = probs[:, 1].detach().cpu().numpy()

pred_df = pd.DataFrame({
    "node_id": np.arange(data.num_nodes),
    "fraud_probability": fraud_probs
})

node_mapping["node_id"] = node_mapping["node_id"].astype(int)

pred_df = pred_df.merge(
    node_mapping,
    on="node_id",
    how="left"
)

# Mark whether the node was labelled or unlabeled
y_cpu = data.y.detach().cpu().numpy()

pred_df["known_label"] = y_cpu
pred_df["is_labelled"] = pred_df["known_label"] != -1

# Add thresholded prediction
pred_df["predicted_label_thresholded"] = (
    pred_df["fraud_probability"] >= best_threshold
).astype(int)

pred_df["predicted_label_name"] = pred_df[
    "predicted_label_thresholded"
].map({
    0: "normal",
    1: "fraud"
})

pred_df = pred_df.sort_values(
    "fraud_probability",
    ascending=False
)

PRED_OUTPUT_PATH = os.path.join(
    OUTPUT_DIR,
    "graphsage_all_node_predictions_thresholded.csv"
)

pred_df.to_csv(
    PRED_OUTPUT_PATH,
    index=False
)

print("\nSaved predictions:")
print(PRED_OUTPUT_PATH)

display(pred_df.head(20))

Data(x=[1828910, 20], edge_index=[2, 6092594], edge_attr=[6092594, 2], num_nodes=1828910)
label
0    45000
1     4466
Name: count, dtype: int64
Total labelled nodes: 49466
Unlabelled nodes: 1779444
Normal labelled nodes: 45000
Fraud labelled nodes: 4466
Train nodes: 34626
Validation nodes: 7420
Test nodes: 7420
Using device: cuda
Class weights: tensor([0.1806, 1.8194], device='cuda:0')
Epoch 010 | Loss: 0.5839 | Train Acc: 0.6616 | Train PR-AUC: 0.5281 | Val Acc: 0.6569 | Val ROC-AUC: 0.9087 | Val PR-AUC: 0.5429
Saved best model.
Epoch 020 | Loss: 0.4536 | Train Acc: 0.8170 | Train PR-AUC: 0.7888 | Val Acc: 0.8106 | Val ROC-AUC: 0.9573 | Val PR-AUC: 0.7914
Saved best model.
Epoch 030 | Loss: 0.3709 | Train Acc: 0.8573 | Train PR-AUC: 0.8493 | Val Acc: 0.8509 | Val ROC-AUC: 0.9700 | Val PR-AUC: 0.8476
Saved best model.
Epoch 040 | Loss: 0.3144 | Train Acc: 0.8937 | Train PR-AUC: 0.8755 | Val Acc: 0.8887 | Val ROC-AUC: 0.9785 | Val PR-AUC: 0.8702
Saved best model.
Epoch 050 | Loss: 0.272

,node_id,fraud_probability,address,known_label,is_labelled,predicted_label_thresholded,predicted_label_name
5605,5605,0.999975,0xb5d85cbf7cb3ee0d56b3bb207d5fc4b82f43f511,1,True,1,fraud
1625562,1625562,0.999967,0xa9d1e08c7793af67e9d92fe308d5697fb81d3e43,1,True,1,fraud
80,80,0.999951,0xa1abfa21f80ecf401bd41365adbb6fef6fefdf09,1,True,1,fraud
29562,29562,0.999941,0xcffad3200574698b78f32232aa9d63eabd290703,1,True,1,fraud
27,27,0.999938,0x28c6c06298d514db089934071355e5743bf21d60,1,True,1,fraud
512,512,0.999931,0xf30ba13e4b04ce5dc4d254ae5fa95477800f0eb0,-1,False,1,fraud
1090,1090,0.999923,0xa9ac43f5b5e38155a288d1a01d2cbc4478e14573,-1,False,1,fraud
1573,1573,0.999916,0x017a71c00d41caf9088a718093874bb069436a79,1,True,1,fraud
1806,1806,0.999914,0xa26148ae51fa8e787df319c04137602cc018b521,1,True,1,fraud
1625561,1625561,0.999903,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,1,True,1,fraud


=========================================================
# GraphSAGE semi-supervised node classification #3
 =========================================================

## We modified the negative sampling strategy to increase the diversity and realism of the normal class. In addition to community-based hard negatives, we included high-activity accounts and randomly sampled active nodes with out-degree greater than one.

In [ ]:
# Fraud nodes:
#   - all nodes in merged_fraud_accounts.csv
#
# Normal nodes:
#   1. 15,000 nodes from the same Infomap communities as fraud nodes,
#      excluding direct neighbors of fraud nodes
#   2. 10,000 high-activity nodes
#   3. 20,000 random nodes from the rest of the graph with out_degree > 1
#
# Output:
#   gnn_labels.csv


# ---------------------------------------------------------
# 1. Paths and parameters
# ---------------------------------------------------------

OUTPUT_DIR = "/content/drive/MyDrive/Graph_Mining_Project/ethereum_gnn/"

NODE_MAPPING_PATH = os.path.join(OUTPUT_DIR, "node_mapping.csv")
NODE_FEATURES_PATH = os.path.join(OUTPUT_DIR, "node_features.csv")
AGG_EDGES_PATH = os.path.join(OUTPUT_DIR, "aggregated_edges.csv")
FRAUD_PATH = os.path.join(OUTPUT_DIR, "merged_fraud_accounts.csv")

INFOMAP_PATH = "/content/drive/MyDrive/Graph_Mining_Project/infomap_communities.csv"

TARGET_COMMUNITY_NORMALS = 15000
TARGET_HIGH_ACTIVITY_NORMALS = 10000
TARGET_RANDOM_NORMALS = 20000

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# ---------------------------------------------------------
# 2. Load files
# ---------------------------------------------------------

node_mapping = pd.read_csv(NODE_MAPPING_PATH)
node_features = pd.read_csv(NODE_FEATURES_PATH)
edges = pd.read_csv(AGG_EDGES_PATH)
fraud_df = pd.read_csv(FRAUD_PATH)
infomap_df = pd.read_csv(INFOMAP_PATH)

# ---------------------------------------------------------
# 3. Standardize columns
# ---------------------------------------------------------

node_mapping["address"] = (
    node_mapping["address"]
    .astype(str)
    .str.lower()
    .str.strip()
)

node_mapping["node_id"] = node_mapping["node_id"].astype(int)

edges["src"] = edges["src"].astype(int)
edges["dst"] = edges["dst"].astype(int)

if "address" in fraud_df.columns:
    fraud_df["address"] = (
        fraud_df["address"]
        .astype(str)
        .str.lower()
        .str.strip()
    )

if "address" in infomap_df.columns:
    infomap_df["address"] = (
        infomap_df["address"]
        .astype(str)
        .str.lower()
        .str.strip()
    )

# Ensure node_features has node_id
if "node_id" not in node_features.columns:
    node_features["node_id"] = node_features.index

node_features["node_id"] = node_features["node_id"].astype(int)

print("Nodes in graph:", len(node_mapping))
print("Aggregated edges:", len(edges))
print("Node features:", node_features.shape)
print("Merged fraud accounts:", len(fraud_df))
print("Infomap rows:", len(infomap_df))

# ---------------------------------------------------------
# 4. Get fraud node IDs
# ---------------------------------------------------------

if "node_id" in fraud_df.columns:
    fraud_df["node_id"] = pd.to_numeric(
        fraud_df["node_id"],
        errors="coerce"
    )
    fraud_df = fraud_df.dropna(subset=["node_id"])
    fraud_df["node_id"] = fraud_df["node_id"].astype(int)

else:
    fraud_df = fraud_df.merge(
        node_mapping[["address", "node_id"]],
        on="address",
        how="inner"
    )

fraud_nodes = set(fraud_df["node_id"].astype(int))

print("Fraud nodes found in graph:", len(fraud_nodes))

# ---------------------------------------------------------
# 5. Attach node_id to Infomap table
# ---------------------------------------------------------

if "node_id" not in infomap_df.columns:
    infomap_df = infomap_df.merge(
        node_mapping[["address", "node_id"]],
        on="address",
        how="inner"
    )

infomap_df["node_id"] = infomap_df["node_id"].astype(int)

print("Infomap nodes found in graph:", len(infomap_df))

# ---------------------------------------------------------
# 6. Compute direct neighbors of fraud nodes
# ---------------------------------------------------------

fraud_out_edges = edges[edges["src"].isin(fraud_nodes)]
fraud_in_edges = edges[edges["dst"].isin(fraud_nodes)]

direct_neighbors_of_fraud = set(fraud_out_edges["dst"]).union(
    set(fraud_in_edges["src"])
)

print("Direct neighbors of fraud nodes:", len(direct_neighbors_of_fraud))

# ---------------------------------------------------------
# 7. Find Infomap communities containing fraud nodes
# ---------------------------------------------------------

fraud_communities = set(
    infomap_df.loc[
        infomap_df["node_id"].isin(fraud_nodes),
        "infomap_id"
    ]
)

nodes_in_fraud_communities = set(
    infomap_df.loc[
        infomap_df["infomap_id"].isin(fraud_communities),
        "node_id"
    ]
)

print("Fraud-related Infomap communities:", len(fraud_communities))
print("Nodes in fraud-related communities:", len(nodes_in_fraud_communities))

# ---------------------------------------------------------
# 8. Sample 15,000 community-based normal nodes
# ---------------------------------------------------------

community_normal_candidates = (
    nodes_in_fraud_communities
    - fraud_nodes
    - direct_neighbors_of_fraud
)

community_normal_candidates = np.array(
    list(community_normal_candidates),
    dtype=int
)

print("Community normal candidates:", len(community_normal_candidates))

if len(community_normal_candidates) < TARGET_COMMUNITY_NORMALS:
    raise ValueError(
        f"Only {len(community_normal_candidates)} community normal candidates found. "
        f"Need {TARGET_COMMUNITY_NORMALS}."
    )

community_normal_nodes = set(
    rng.choice(
        community_normal_candidates,
        size=TARGET_COMMUNITY_NORMALS,
        replace=False
    )
)

print("Selected community normals:", len(community_normal_nodes))

# ---------------------------------------------------------
# 9. Sample 10,000 high-activity normal nodes
# ---------------------------------------------------------
# High activity is defined using total_tx if available.
# If total_tx is not available, it is approximated with in_degree + out_degree.

features = node_features.copy()

if "total_tx" not in features.columns:
    if {"in_degree", "out_degree"}.issubset(features.columns):
        features["total_tx"] = features["in_degree"] + features["out_degree"]
    else:
        raise ValueError(
            "node_features must contain either 'total_tx' or both 'in_degree' and 'out_degree'."
        )

all_graph_nodes = set(node_mapping["node_id"].astype(int))

excluded_before_high_activity = (
    fraud_nodes
    | direct_neighbors_of_fraud
    | community_normal_nodes
)

high_activity_candidates = features[
    ~features["node_id"].isin(excluded_before_high_activity)
].copy()

# Keep only nodes that exist in the graph
high_activity_candidates = high_activity_candidates[
    high_activity_candidates["node_id"].isin(all_graph_nodes)
]

# Sort by activity descending
high_activity_candidates = high_activity_candidates.sort_values(
    "total_tx",
    ascending=False
)

print("High-activity candidates:", len(high_activity_candidates))

if len(high_activity_candidates) < TARGET_HIGH_ACTIVITY_NORMALS:
    raise ValueError(
        f"Only {len(high_activity_candidates)} high-activity candidates found. "
        f"Need {TARGET_HIGH_ACTIVITY_NORMALS}."
    )

high_activity_normal_nodes = set(
    high_activity_candidates
    .head(TARGET_HIGH_ACTIVITY_NORMALS)["node_id"]
    .astype(int)
)

print("Selected high-activity normals:", len(high_activity_normal_nodes))

# ---------------------------------------------------------
# 10. Sample 20,000 random normal nodes with out_degree > 1
# ---------------------------------------------------------

excluded_before_random = (
    fraud_nodes
    | direct_neighbors_of_fraud
    | community_normal_nodes
    | high_activity_normal_nodes
)

if "out_degree" not in features.columns:
    # Compute unique-neighbor out_degree from aggregated_edges.csv
    out_degree_df = (
        edges.groupby("src")["dst"]
        .nunique()
        .reset_index()
        .rename(columns={"src": "node_id", "dst": "out_degree"})
    )

    features = features.drop(columns=["out_degree"], errors="ignore")
    features = features.merge(
        out_degree_df,
        on="node_id",
        how="left"
    )

    features["out_degree"] = features["out_degree"].fillna(0)

random_candidates = features[
    (features["out_degree"] > 1)
    & (~features["node_id"].isin(excluded_before_random))
].copy()

random_candidates = random_candidates[
    random_candidates["node_id"].isin(all_graph_nodes)
]

random_candidate_nodes = np.array(
    random_candidates["node_id"].astype(int).unique(),
    dtype=int
)

print("Random candidates with out_degree > 1:", len(random_candidate_nodes))

if len(random_candidate_nodes) < TARGET_RANDOM_NORMALS:
    raise ValueError(
        f"Only {len(random_candidate_nodes)} random candidates with out_degree > 1 found. "
        f"Need {TARGET_RANDOM_NORMALS}."
    )

random_normal_nodes = set(
    rng.choice(
        random_candidate_nodes,
        size=TARGET_RANDOM_NORMALS,
        replace=False
    )
)

print("Selected random normals:", len(random_normal_nodes))

# ---------------------------------------------------------
# 11. Create final label table
# ---------------------------------------------------------

fraud_labels = pd.DataFrame({
    "node_id": list(fraud_nodes),
    "label": 1,
    "label_type": "fraud_merged"
})

community_normal_labels = pd.DataFrame({
    "node_id": list(community_normal_nodes),
    "label": 0,
    "label_type": "normal_same_fraud_infomap_community_not_direct_neighbor"
})

high_activity_normal_labels = pd.DataFrame({
    "node_id": list(high_activity_normal_nodes),
    "label": 0,
    "label_type": "normal_high_activity"
})

random_normal_labels = pd.DataFrame({
    "node_id": list(random_normal_nodes),
    "label": 0,
    "label_type": "normal_random_out_degree_gt_1"
})

gnn_labels = pd.concat(
    [
        fraud_labels,
        community_normal_labels,
        high_activity_normal_labels,
        random_normal_labels
    ],
    ignore_index=True
)

# Safety: remove accidental duplicate node_ids.
# If a duplicate exists, fraud label dominates.
gnn_labels = gnn_labels.sort_values(
    "label",
    ascending=False
).drop_duplicates(
    subset=["node_id"],
    keep="first"
)

gnn_labels = gnn_labels.merge(
    node_mapping[["node_id", "address"]],
    on="node_id",
    how="left"
)

gnn_labels = gnn_labels[
    ["node_id", "address", "label", "label_type"]
]

print("\nFinal label counts:")
print(gnn_labels["label"].value_counts())

print("\nLabel type counts:")
print(gnn_labels["label_type"].value_counts())

print("\nTotal labelled nodes:", len(gnn_labels))

# ---------------------------------------------------------
# 12. Save final labels
# ---------------------------------------------------------

LABEL_OUTPUT_PATH = os.path.join(
    OUTPUT_DIR,
    "gnn_labels_02.csv"
)

gnn_labels.to_csv(
    LABEL_OUTPUT_PATH,
    index=False
)

print("\nSaved:", LABEL_OUTPUT_PATH)

Nodes in graph: 1828910
Aggregated edges: 2948955
Node features: (1828910, 21)
Merged fraud accounts: 4466
Infomap rows: 1686534
Fraud nodes found in graph: 4466
Infomap nodes found in graph: 1686380
Direct neighbors of fraud nodes: 1113021
Fraud-related Infomap communities: 2586
Nodes in fraud-related communities: 1125121
Community normal candidates: 198271
Selected community normals: 15000
High-activity candidates: 699292
Selected high-activity normals: 10000
Random candidates with out_degree > 1: 93092
Selected random normals: 20000

Final label counts:
label
0    45000
1     4466
Name: count, dtype: int64

Label type counts:
label_type
normal_random_out_degree_gt_1                              20000
normal_same_fraud_infomap_community_not_direct_neighbor    15000
normal_high_activity                                       10000
fraud_merged                                                4466
Name: count, dtype: int64

Total labelled nodes: 49466

Saved: /content/drive/MyDrive/Graph_

In [ ]:
# =========================================================
# GraphSAGE semi-supervised node classification
# with validation-selected fraud threshold
# =========================================================

# ---------------------------------------------------------
# 1. Paths
# ---------------------------------------------------------

OUTPUT_DIR = "/content/drive/MyDrive/Graph_Mining_Project/ethereum_gnn/"
GRAPH_PATH = os.path.join(OUTPUT_DIR, "ethereum_full_graph.pt")
LABEL_PATH = os.path.join(OUTPUT_DIR, "gnn_labels_02.csv")
NODE_MAPPING_PATH = os.path.join(OUTPUT_DIR, "node_mapping.csv")

# ---------------------------------------------------------
# 2. Load graph and labels
# ---------------------------------------------------------

data = torch.load(
    GRAPH_PATH,
    map_location="cpu",
    weights_only=False
)

labels_df = pd.read_csv(LABEL_PATH)
node_mapping = pd.read_csv(NODE_MAPPING_PATH)

labels_df["node_id"] = labels_df["node_id"].astype(int)
labels_df["label"] = labels_df["label"].astype(int)

print(data)
print(labels_df["label"].value_counts())
print("Total labelled nodes:", len(labels_df))

# ---------------------------------------------------------
# 3. Create y vector
# ---------------------------------------------------------
# -1 = unlabeled
#  0 = normal
#  1 = fraud

num_nodes = data.num_nodes

y = torch.full(
    (num_nodes,),
    -1,
    dtype=torch.long
)

labelled_node_ids = labels_df["node_id"].values
label_values = labels_df["label"].values

y[labelled_node_ids] = torch.tensor(
    label_values,
    dtype=torch.long
)

data.y = y

print("Unlabelled nodes:", (data.y == -1).sum().item())
print("Normal labelled nodes:", (data.y == 0).sum().item())
print("Fraud labelled nodes:", (data.y == 1).sum().item())

# ---------------------------------------------------------
# 4. Train / validation / test split
# ---------------------------------------------------------

train_ids, temp_ids = train_test_split(
    labelled_node_ids,
    test_size=0.30,
    random_state=42,
    stratify=label_values
)

temp_labels = y[temp_ids].numpy()

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)

train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)

train_mask[train_ids] = True
val_mask[val_ids] = True
test_mask[test_ids] = True

data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

print("Train nodes:", data.train_mask.sum().item())
print("Validation nodes:", data.val_mask.sum().item())
print("Test nodes:", data.test_mask.sum().item())

# ---------------------------------------------------------
# 5. Define GraphSAGE model
# ---------------------------------------------------------

class GraphSAGE(torch.nn.Module):
    def __init__(
        self,
        in_channels,
        hidden_channels,
        out_channels,
        dropout=0.3
    ):
        super().__init__()

        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )
        x = self.conv2(x, edge_index)
        return x

# ---------------------------------------------------------
# 6. Device, model, optimizer
# ---------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

data = data.to(device)

model = GraphSAGE(
    in_channels=data.x.shape[1],
    hidden_channels=64,
    out_channels=2,
    dropout=0.3
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=5e-4
)


# ---------------------------------------------------------
# 7. Class weights for imbalance
# ---------------------------------------------------------

train_labels = data.y[data.train_mask]

num_normal = (train_labels == 0).sum().item()
num_fraud = (train_labels == 1).sum().item()

class_weights = torch.tensor(
    [
        1.0 / num_normal,
        1.0 / num_fraud
    ],
    dtype=torch.float,
    device=device
)

class_weights = class_weights / class_weights.sum() * 2

print("Class weights:", class_weights)
# ---------------------------------------------------------
# 8. Training and evaluation functions
# ---------------------------------------------------------

def train_one_epoch():
    model.train()
    optimizer.zero_grad()

    out = model(data.x, data.edge_index)

    loss = F.cross_entropy(
        out[data.train_mask],
        data.y[data.train_mask],
        weight=class_weights
    )

    loss.backward()
    optimizer.step()

    return loss.item()


@torch.no_grad()
def evaluate(mask):
    model.eval()

    out = model(data.x, data.edge_index)

    logits = out[mask]
    labels = data.y[mask]

    probs = F.softmax(logits, dim=1)[:, 1]
    preds_argmax = logits.argmax(dim=1)

    acc = (preds_argmax == labels).float().mean().item()

    probs_np = probs.detach().cpu().numpy()
    preds_np = preds_argmax.detach().cpu().numpy()
    labels_np = labels.detach().cpu().numpy()

    roc_auc = roc_auc_score(labels_np, probs_np)
    pr_auc = average_precision_score(labels_np, probs_np)

    return acc, roc_auc, pr_auc, preds_np, probs_np, labels_np

# ---------------------------------------------------------
# 9. Train model
# ---------------------------------------------------------

EPOCHS = 150

best_val_pr_auc = 0
best_model_path = os.path.join(OUTPUT_DIR, "best_graphsage_model_03.pt")

for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch()

    if epoch % 10 == 0:
        train_acc, train_roc, train_pr, _, _, _ = evaluate(data.train_mask)
        val_acc, val_roc, val_pr, _, _, _ = evaluate(data.val_mask)

        print(
            f"Epoch {epoch:03d} | "
            f"Loss: {loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Train PR-AUC: {train_pr:.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"Val ROC-AUC: {val_roc:.4f} | "
            f"Val PR-AUC: {val_pr:.4f}"
        )

        if val_pr > best_val_pr_auc:
            best_val_pr_auc = val_pr

            torch.save(
                model.state_dict(),
                best_model_path
            )

            print("Saved best model.")

# ---------------------------------------------------------
# 10. Load best model
# ---------------------------------------------------------

model.load_state_dict(
    torch.load(
        best_model_path,
        map_location=device
    )
)

# ---------------------------------------------------------
# 11. Select best fraud threshold on validation set
# ---------------------------------------------------------
# Choose the threshold that maximizes F1-score on validation data.

val_acc, val_roc, val_pr, val_preds_argmax, val_probs, val_labels = evaluate(
    data.val_mask
)

thresholds = np.arange(0.05, 0.96, 0.01)

threshold_results = []

for threshold in thresholds:
    val_preds_thresholded = (val_probs >= threshold).astype(int)

    precision = precision_score(
        val_labels,
        val_preds_thresholded,
        zero_division=0
    )

    recall = recall_score(
        val_labels,
        val_preds_thresholded,
        zero_division=0
    )

    f1 = f1_score(
        val_labels,
        val_preds_thresholded,
        zero_division=0
    )

    threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_results = pd.DataFrame(threshold_results)

best_row = threshold_results.loc[
    threshold_results["f1"].idxmax()
]

best_threshold = float(best_row["threshold"])

print("\nBest threshold selected on validation set:")
print(best_row)

# Save threshold search results
threshold_results_path = os.path.join(
    OUTPUT_DIR,
    "graphsage_threshold_results_validation_03.csv"
)

threshold_results.to_csv(
    threshold_results_path,
    index=False
)

print("Saved threshold results:", threshold_results_path)

# ---------------------------------------------------------
# 12. Final test evaluation with selected threshold
# ---------------------------------------------------------

test_acc_argmax, test_roc, test_pr, test_preds_argmax, test_probs, test_labels = evaluate(
    data.test_mask
)

test_preds_thresholded = (test_probs >= best_threshold).astype(int)

print("\nFinal test results with validation-selected threshold")
print("Selected threshold:", best_threshold)
print("Test ROC-AUC:", test_roc)
print("Test PR-AUC:", test_pr)

print("\nClassification report:")
print(
    classification_report(
        test_labels,
        test_preds_thresholded,
        target_names=["normal", "fraud"],
        zero_division=0
    )
)

print("\nConfusion matrix:")
print(confusion_matrix(test_labels, test_preds_thresholded))

# ---------------------------------------------------------
# 13. Predict fraud probability for every node
# ---------------------------------------------------------

model.eval()

with torch.no_grad():
    out = model(data.x, data.edge_index)
    probs = F.softmax(out, dim=1)
    fraud_probs = probs[:, 1].detach().cpu().numpy()

pred_df = pd.DataFrame({
    "node_id": np.arange(data.num_nodes),
    "fraud_probability": fraud_probs
})

node_mapping["node_id"] = node_mapping["node_id"].astype(int)

pred_df = pred_df.merge(
    node_mapping,
    on="node_id",
    how="left"
)

# Mark whether the node was labelled or unlabeled
y_cpu = data.y.detach().cpu().numpy()

pred_df["known_label"] = y_cpu
pred_df["is_labelled"] = pred_df["known_label"] != -1

# Add thresholded prediction
pred_df["predicted_label_thresholded"] = (
    pred_df["fraud_probability"] >= best_threshold
).astype(int)

pred_df["predicted_label_name"] = pred_df[
    "predicted_label_thresholded"
].map({
    0: "normal",
    1: "fraud"
})

pred_df = pred_df.sort_values(
    "fraud_probability",
    ascending=False
)

PRED_OUTPUT_PATH = os.path.join(
    OUTPUT_DIR,
    "graphsage_all_node_predictions_New_labels.csv"
)

pred_df.to_csv(
    PRED_OUTPUT_PATH,
    index=False
)

print("\nSaved predictions:")
print(PRED_OUTPUT_PATH)

display(pred_df.head(20))

Data(x=[1828910, 20], edge_index=[2, 6092594], edge_attr=[6092594, 2], num_nodes=1828910)
label
0    45000
1     4466
Name: count, dtype: int64
Total labelled nodes: 49466
Unlabelled nodes: 1779444
Normal labelled nodes: 45000
Fraud labelled nodes: 4466
Train nodes: 34626
Validation nodes: 7420
Test nodes: 7420
Using device: cuda
Class weights: tensor([0.1806, 1.8194], device='cuda:0')
Epoch 010 | Loss: 0.5923 | Train Acc: 0.6062 | Train PR-AUC: 0.4787 | Val Acc: 0.6008 | Val ROC-AUC: 0.8643 | Val PR-AUC: 0.4980
Saved best model.
Epoch 020 | Loss: 0.4925 | Train Acc: 0.6677 | Train PR-AUC: 0.6112 | Val Acc: 0.6620 | Val ROC-AUC: 0.9020 | Val PR-AUC: 0.6196
Saved best model.
Epoch 030 | Loss: 0.4380 | Train Acc: 0.7811 | Train PR-AUC: 0.6694 | Val Acc: 0.7796 | Val ROC-AUC: 0.9254 | Val PR-AUC: 0.6678
Saved best model.
Epoch 040 | Loss: 0.3906 | Train Acc: 0.8186 | Train PR-AUC: 0.6875 | Val Acc: 0.8163 | Val ROC-AUC: 0.9362 | Val PR-AUC: 0.6806
Saved best model.
Epoch 050 | Loss: 0.351

,node_id,fraud_probability,address,known_label,is_labelled,predicted_label_thresholded,predicted_label_name
5605,5605,0.999964,0xb5d85cbf7cb3ee0d56b3bb207d5fc4b82f43f511,1,True,1,fraud
1806,1806,0.999954,0xa26148ae51fa8e787df319c04137602cc018b521,1,True,1,fraud
980,980,0.999950,0x974caa59e49682cda0ad2bbe82983419a2ecc400,1,True,1,fraud
80,80,0.999949,0xa1abfa21f80ecf401bd41365adbb6fef6fefdf09,1,True,1,fraud
27,27,0.999940,0x28c6c06298d514db089934071355e5743bf21d60,1,True,1,fraud
512,512,0.999940,0xf30ba13e4b04ce5dc4d254ae5fa95477800f0eb0,-1,False,1,fraud
1573,1573,0.999935,0x017a71c00d41caf9088a718093874bb069436a79,1,True,1,fraud
4260,4260,0.999922,0x53ffd4cc068dad23e1016daaa2706679fe884ca6,1,True,1,fraud
217035,217035,0.999912,0x89214882e9a2764fc0ad1325fbf8e9cbe11e1063,1,True,1,fraud
46,46,0.999897,0x1ab4973a48dc892cd9971ece8e01dcc7688f8f23,1,True,1,fraud


In [ ]:

# ---------------------------------------------------------
# Path
# ---------------------------------------------------------

PRED_PATH = "/content/drive/MyDrive/Graph_Mining_Project/ethereum_gnn/graphsage_all_node_predictions_New_labels.csv"

# ---------------------------------------------------------
# Load predictions
# ---------------------------------------------------------

pred_df = pd.read_csv(PRED_PATH)

print(pred_df.columns)

# ---------------------------------------------------------
# Count fraud predictions
# ---------------------------------------------------------

# If your file contains thresholded predictions
# created with the previous code:

fraud_count = (
    pred_df["predicted_label_thresholded"] == 1
).sum()

total_nodes = len(pred_df)

fraud_percentage = 100 * fraud_count / total_nodes

print("Total nodes:", total_nodes)
print("Predicted fraud nodes:", fraud_count)
print(f"Percentage predicted as fraud: {fraud_percentage:.4f}%")

Index(['node_id', 'fraud_probability', 'address', 'known_label', 'is_labelled',
       'predicted_label_thresholded', 'predicted_label_name'],
      dtype='object')
Total nodes: 1828910
Predicted fraud nodes: 14485
Percentage predicted as fraud: 0.7920%


# We try to add Time Motifs


In [ ]:
# =========================================================
# Temporal motif counting for ALL graph nodes
# =========================================================
#
# This code computes simple directed temporal motifs
# over a rolling 1-hour window for every node.
#
# Motifs counted:
#
# 1. temporal_out_star
# 2. temporal_in_star
# 3. temporal_chain
# 4. temporal_reciprocal
# 5. temporal_cycle_3
# 6. temporal_fan_in_out
#
# The goal is NOT perfect motif mining efficiency,
# but rather:
#
# - obtain motif features for all nodes
# - test runtime feasibility
# - create node-level temporal motif vectors
#
# =========================================================

import time

# ---------------------------------------------------------
# 1. Parameters
# ---------------------------------------------------------

TX_PATH = "/content/drive/MyDrive/Graph_Mining_Project/eth_tx_last4days.csv"

OUTPUT_DIR = "/content/drive/MyDrive/Graph_Mining_Project/ethereum_gnn/"

OUTPUT_PATH = os.path.join(
    OUTPUT_DIR,
    "gnn_temporal_motif_features_1h_total_ALL_NODES.csv"
)

TIME_WINDOW_SECONDS = 3600   # 1 hour

# ---------------------------------------------------------
# 2. Load transactions
# ---------------------------------------------------------

print("Loading transactions...")

tx = pd.read_csv(
    TX_PATH,
    usecols=[
        "from_address",
        "to_address",
        "block_timestamp"
    ]
)

tx = tx.dropna()

tx["from_address"] = (
    tx["from_address"]
    .astype(str)
    .str.lower()
    .str.strip()
)

tx["to_address"] = (
    tx["to_address"]
    .astype(str)
    .str.lower()
    .str.strip()
)

tx["block_timestamp"] = pd.to_datetime(
    tx["block_timestamp"],
    errors="coerce",
    utc=True
)

tx = tx.dropna(subset=["block_timestamp"])

tx["timestamp_unix"] = (
    tx["block_timestamp"].astype("int64") // 10**9
)

tx["timestamp_unix"] = tx["timestamp_unix"].astype(np.int64)

print("Transactions loaded:", len(tx))

# ---------------------------------------------------------
# 3. Sort by time
# ---------------------------------------------------------

tx = tx.sort_values("timestamp_unix").reset_index(drop=True)


Loading transactions...
Transactions loaded: 6092594


In [ ]:

# ---------------------------------------------------------
# 4. Create temporal adjacency structures
# ---------------------------------------------------------

print("Creating temporal structures...")

# outgoing[src] -> list of (dst, timestamp)
outgoing = defaultdict(list)

# incoming[dst] -> list of (src, timestamp)
incoming = defaultdict(list)

for row in tqdm(
    tx.itertuples(index=False),
    total=len(tx),
    desc="Building temporal structures"
):
    src = row.from_address
    dst = row.to_address
    ts = row.timestamp_unix

    outgoing[src].append((dst, ts))
    incoming[dst].append((src, ts))

# ---------------------------------------------------------
# 5. Initialize motif counters
# ---------------------------------------------------------

motif_counts = defaultdict(
    lambda: {
        "temporal_out_star": 0,
        "temporal_in_star": 0,
        "temporal_chain": 0,
        "temporal_reciprocal": 0,
        "temporal_cycle_3": 0
    }
)


Creating temporal structures...


Building temporal structures:   0%|          | 0/6092594 [00:00<?, ?it/s]

In [ ]:

# ---------------------------------------------------------
# Temporal OUT-STAR with sliding window
# ---------------------------------------------------------

print("\nCounting temporal_out_star with sliding window...")

start = time.time()

for node, events in tqdm(
    outgoing.items(),
    total=len(outgoing),
    desc="OUT-STAR"
):
    events = sorted(events, key=lambda x: x[1])
    left = 0

    for right in range(len(events)):
        while events[right][1] - events[left][1] > TIME_WINDOW_SECONDS:
            left += 1

        window_size = right - left

        motif_counts[node]["temporal_out_star"] += window_size

end = time.time()
print(f"OUT-STAR completed in {(end-start)/60:.2f} minutes")


Counting temporal_out_star with sliding window...


OUT-STAR:   0%|          | 0/1625532 [00:00<?, ?it/s]

OUT-STAR completed in 0.14 minutes


In [ ]:

# ---------------------------------------------------------
# 7. Temporal IN-STAR
# ---------------------------------------------------------
#
# B -> A
# C -> A
#
# ---------------------------------------------------------

print("\nCounting temporal_in_star with sliding window...")

start = time.time()

for node, events in tqdm(
    incoming.items(),
    total=len(incoming),
    desc="IN-STAR"
):
    events = sorted(events, key=lambda x: x[1])
    left = 0

    for right in range(len(events)):
        while events[right][1] - events[left][1] > TIME_WINDOW_SECONDS:
            left += 1

        window_size = right - left

        # number of previous incoming events within 1h
        motif_counts[node]["temporal_in_star"] += window_size

end = time.time()
print(f"IN-STAR completed in {(end-start)/60:.2f} minutes")



Counting temporal_in_star with sliding window...


IN-STAR:   0%|          | 0/817935 [00:00<?, ?it/s]

IN-STAR completed in 0.12 minutes


In [ ]:
# ---------------------------------------------------------
# Temporal CHAIN with sliding window
# ---------------------------------------------------------
#
# Pattern:
#   A -> B -> C
#
# The counted node is the middle node B.
# We count cases where B receives a transaction and then sends
# a transaction within TIME_WINDOW_SECONDS.
#
# This is much faster than the naive double loop.

print("\nCounting temporal_chain with sliding window...")

start = time.time()

for node in tqdm(
    set(incoming.keys()).intersection(set(outgoing.keys())),
    desc="CHAIN"
):
    in_events = sorted(incoming[node], key=lambda x: x[1])
    out_events = sorted(outgoing[node], key=lambda x: x[1])

    left = 0
    right = 0
    n_out = len(out_events)

    for src, t_in in in_events:

        # Move left pointer to first outgoing event after or at t_in
        while left < n_out and out_events[left][1] < t_in:
            left += 1

        # Move right pointer to first outgoing event outside the 1h window
        if right < left:
            right = left

        while right < n_out and out_events[right][1] - t_in <= TIME_WINDOW_SECONDS:
            right += 1

        # Outgoing events in [t_in, t_in + TIME_WINDOW_SECONDS]
        window_size = right - left

        motif_counts[node]["temporal_chain"] += window_size

end = time.time()

print(f"CHAIN completed in {(end-start)/60:.2f} minutes")



Counting temporal_chain with sliding window...


CHAIN:   0%|          | 0/614557 [00:00<?, ?it/s]

CHAIN completed in 0.11 minutes


In [ ]:

# ---------------------------------------------------------
# 9. Temporal RECIPROCAL
# ---------------------------------------------------------
#
# A -> B
# B -> A
#
# ---------------------------------------------------------

print("\nCounting temporal_reciprocal...")
start = time.time()
for src in tqdm(
    outgoing.keys(),
    total=len(outgoing),
    desc="RECIPROCAL"
):

    out_events = outgoing[src]

    for dst, t1 in out_events:

        reverse_events = outgoing.get(dst, [])

        for rev_dst, t2 in reverse_events:

            if rev_dst != src:
                continue

            if t2 < t1:
                continue

            if t2 - t1 > TIME_WINDOW_SECONDS:
                continue

            motif_counts[src]["temporal_reciprocal"] += 1
end = time.time()

print(f"RECIPROCAL completed in {(end-start)/60:.2f} minutes")



Counting temporal_reciprocal...


RECIPROCAL:   0%|          | 0/1625532 [00:00<?, ?it/s]

RECIPROCAL completed in 16.15 minutes


In [ ]:


# ---------------------------------------------------------
# Sort temporal adjacency lists once
# ---------------------------------------------------------

print("Sorting temporal adjacency lists...")

for node in tqdm(outgoing.keys(), total=len(outgoing), desc="Sort outgoing"):
    outgoing[node].sort(key=lambda x: x[1])

for node in tqdm(incoming.keys(), total=len(incoming), desc="Sort incoming"):
    incoming[node].sort(key=lambda x: x[1])

out_sizes = pd.Series({node: len(events) for node, events in outgoing.items()})
in_sizes = pd.Series({node: len(events) for node, events in incoming.items()})

print("Outgoing size distribution:")
print(out_sizes.describe())

print("\nTop 20 outgoing nodes:")
print(out_sizes.sort_values(ascending=False).head(20))

print("\nIncoming size distribution:")
print(in_sizes.describe())

print("\nTop 20 incoming nodes:")
print(in_sizes.sort_values(ascending=False).head(20))

Sorting temporal adjacency lists...


Sort outgoing:   0%|          | 0/1625532 [00:00<?, ?it/s]

Sort incoming:   0%|          | 0/817935 [00:00<?, ?it/s]

Outgoing size distribution:
count    1.625532e+06
mean     3.748062e+00
std      1.147887e+02
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      2.000000e+00
max      4.614300e+04
dtype: float64

Top 20 outgoing nodes:
0x28c6c06298d514db089934071355e5743bf21d60    46143
0x559432e18b281731c054cd703d4b49872be4ed53    41888
0x4838b106fce9647bdf1e7877bf73ce8b0bad5f97    41087
0xf70da97812cb96acdf810712aa562db8dfa3dbef    38233
0x974caa59e49682cda0ad2bbe82983419a2ecc400    32590
0x46340b20830761efd32832a74d7169b29feb9758    27745
0x21a31ee1afc51d94c2efccaa2092ad1028285549    26828
0xdfd5293d8e347dfe59e90efd55b2956a1343963d    26495
0x264bd8291fae1d75db2c5f573b07faa6715997b5    24601
0x5050f69a9786f081509234f1a7f4684b5e5b76c9    24108
0xcf076007e6c36cfb1eb2cd36ad66405fec7d0b31    22129
0x1ab4973a48dc892cd9971ece8e01dcc7688f8f23    20927
0x0d0707963952f2fba59dd06f2b425ace40b492fe    18337
0xdadb0d80178819f2319190d340ce9a924f783711    18204
0x646c4fbdf82b5766c5eaf1fab9a

In [ ]:
# ---------------------------------------------------------
# Temporal 3-CYCLE: exact for small nodes, estimated for large nodes
# ---------------------------------------------------------
#
# Pattern:
#   A -> B -> C -> A
#
# Conditions:
#   t1 <= t2 <= t3
#   t3 - t1 <= TIME_WINDOW_SECONDS
#
# Count is attributed to A.
#
# For small nodes:
#   exact count
#
# For large/high-cost nodes:
#   sample outgoing A -> B events and scale up the estimate
#
# The final value is stored in the SAME column:
#   motif_counts[A]["temporal_cycle_3"]
#
# No second feature column is created.

import time
import bisect
import random
from tqdm.notebook import tqdm

print("\nCounting temporal_cycle_3 with hybrid exact/estimated method...")

# ---------------------------------------------------------
# 1. Prepare sorted outgoing lists
# ---------------------------------------------------------

print("Preparing sorted outgoing lists...")

outgoing_sorted = {}
outgoing_times = {}

for node, events in tqdm(
    outgoing.items(),
    total=len(outgoing),
    desc="Preparing outgoing"
):
    events_sorted = sorted(events, key=lambda x: x[1])
    outgoing_sorted[node] = events_sorted
    outgoing_times[node] = [t for _, t in events_sorted]

# ---------------------------------------------------------
# 2. Parameters
# ---------------------------------------------------------

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# If A has more than this many outgoing events, approximate it
MAX_A_OUT_EVENTS_EXACT = 3000

# Number of A->B events sampled for approximate nodes
A_EDGE_SAMPLE_SIZE = 500

# If B or C has a very large outgoing list, sample candidates inside the valid time window
MAX_WINDOW_EVENTS_EXACT = 3000
WINDOW_SAMPLE_SIZE = 500

PRINT_EVERY_NODES = 5000
SLOW_NODE_SECONDS = 60

nodes_to_process = sorted(
    outgoing_sorted.keys(),
    key=lambda n: len(outgoing_sorted[n])
)

cycle_diagnostics = []

cycle_start_time = time.time()

# ---------------------------------------------------------
# 3. Helper: count cycles starting from one event A -> B at t1
# ---------------------------------------------------------

def count_cycles_from_edge(a, b, t1):
    """
    Counts or estimates the number of temporal cycles:

        A -> B -> C -> A

    starting from one fixed edge A -> B at time t1.

    Returns:
        estimated number of cycles starting from this edge
    """

    b_out_events = outgoing_sorted.get(b)

    if not b_out_events:
        return 0.0

    b_times = outgoing_times[b]

    # B -> C must occur in [t1, t1 + window]
    b_left = bisect.bisect_left(b_times, t1)
    b_right = bisect.bisect_right(
        b_times,
        t1 + TIME_WINDOW_SECONDS
    )

    b_window = b_out_events[b_left:b_right]

    if len(b_window) == 0:
        return 0.0

    # If the B-window is huge, sample it and scale up
    if len(b_window) > MAX_WINDOW_EVENTS_EXACT:
        sampled_b_window = random.sample(
            b_window,
            WINDOW_SAMPLE_SIZE
        )
        b_scale = len(b_window) / WINDOW_SAMPLE_SIZE
    else:
        sampled_b_window = b_window
        b_scale = 1.0

    cycle_count = 0.0

    for c, t2 in sampled_b_window:

        c_out_events = outgoing_sorted.get(c)

        if not c_out_events:
            continue

        c_times = outgoing_times[c]

        # C -> A must occur in [t2, t1 + window]
        c_left = bisect.bisect_left(c_times, t2)
        c_right = bisect.bisect_right(
            c_times,
            t1 + TIME_WINDOW_SECONDS
        )

        c_window = c_out_events[c_left:c_right]

        if len(c_window) == 0:
            continue

        # If C-window is huge, sample and scale up
        if len(c_window) > MAX_WINDOW_EVENTS_EXACT:
            sampled_c_window = random.sample(
                c_window,
                WINDOW_SAMPLE_SIZE
            )
            c_scale = len(c_window) / WINDOW_SAMPLE_SIZE
        else:
            sampled_c_window = c_window
            c_scale = 1.0

        hits = 0

        for dst, t3 in sampled_c_window:
            if dst == a:
                hits += 1

        cycle_count += b_scale * c_scale * hits

    return cycle_count

# ---------------------------------------------------------
# 4. Main loop
# ---------------------------------------------------------

for idx, a in enumerate(tqdm(
    nodes_to_process,
    total=len(nodes_to_process),
    desc="CYCLE_3"
)):
    node_start_time = time.time()

    a_out_events = outgoing_sorted[a]
    a_out_count = len(a_out_events)

    if idx % PRINT_EVERY_NODES == 0:
        print(
            f"\nProcessing idx={idx}/{len(nodes_to_process)} | "
            f"address={a} | "
            f"a_out={a_out_count} | "
            f"a_in={len(incoming.get(a, []))}"
        )

    # -----------------------------------------------------
    # Exact mode
    # -----------------------------------------------------

    if a_out_count <= MAX_A_OUT_EVENTS_EXACT:
        sampled_a_events = a_out_events
        a_scale = 1.0
        mode = "exact"

    # -----------------------------------------------------
    # Approximate mode
    # -----------------------------------------------------

    else:
        sample_size = min(A_EDGE_SAMPLE_SIZE, a_out_count)

        sampled_a_events = random.sample(
            a_out_events,
            sample_size
        )

        a_scale = a_out_count / sample_size
        mode = "estimated"

    cycle_count_for_a = 0.0

    for b, t1 in sampled_a_events:
        cycle_count_for_a += count_cycles_from_edge(a, b, t1)

    # Scale up if A was sampled
    cycle_count_for_a = cycle_count_for_a * a_scale

    # Store in the same motif column
    motif_counts[a]["temporal_cycle_3"] += cycle_count_for_a

    node_elapsed = time.time() - node_start_time

    if mode == "estimated" or node_elapsed > SLOW_NODE_SECONDS:
        cycle_diagnostics.append({
            "idx": idx,
            "address": a,
            "mode": mode,
            "seconds": node_elapsed,
            "a_out": a_out_count,
            "a_in": len(incoming.get(a, [])),
            "cycle_3_value": cycle_count_for_a
        })

        print(
            f"\n{mode.upper()} node | "
            f"idx={idx} | "
            f"address={a} | "
            f"time={node_elapsed:.2f}s | "
            f"a_out={a_out_count} | "
            f"cycle_3={cycle_count_for_a:.2f}"
        )

# ---------------------------------------------------------
# 5. Summary
# ---------------------------------------------------------

total_elapsed = time.time() - cycle_start_time

print(f"\nCYCLE_3 completed in {total_elapsed / 60:.2f} minutes")

cycle_diagnostics_df = pd.DataFrame(cycle_diagnostics)

if len(cycle_diagnostics_df) > 0:
    print("\nNodes estimated or slow:")
    display(
        cycle_diagnostics_df.sort_values(
            "cycle_3_value",
            ascending=False
        ).head(20)
    )

    diagnostics_path = os.path.join(
        OUTPUT_DIR,
        "cycle_3_estimation_diagnostics.csv"
    )

    cycle_diagnostics_df.to_csv(
        diagnostics_path,
        index=False
    )

    print("Saved diagnostics:", diagnostics_path)


Counting temporal_cycle_3 with hybrid exact/estimated method...
Preparing sorted outgoing lists...


Preparing outgoing:   0%|          | 0/1625532 [00:00<?, ?it/s]

CYCLE_3:   0%|          | 0/1625532 [00:00<?, ?it/s]


Processing idx=0/1625532 | address=0x89211817d39ca280220727bf0a1b1cbb2178915d | a_out=1 | a_in=0

Processing idx=5000/1625532 | address=0xd3227d1abcb5fbb17cc491a8ec5f12b03fb2df86 | a_out=1 | a_in=1

Processing idx=10000/1625532 | address=0x72b1121248a25ff3bd9e4a2fcf6b5e8bfc39c44e | a_out=1 | a_in=0

Processing idx=15000/1625532 | address=0x1bce7732df3769884a04fd6c45136a499b4c0148 | a_out=1 | a_in=0

Processing idx=20000/1625532 | address=0x5dbc65a9ab5b700a7247993cfeed07a7b8f22f25 | a_out=1 | a_in=1

Processing idx=25000/1625532 | address=0x58de44b5cc742f44bab0ca0a0162b0a54c629db8 | a_out=1 | a_in=1

Processing idx=30000/1625532 | address=0xf5d74c50bdceeed7a116550146ebefccb946b1f9 | a_out=1 | a_in=0

Processing idx=35000/1625532 | address=0xd610c2c3f0160e46ab9ee5028cbfbf6908d0f985 | a_out=1 | a_in=0

Processing idx=40000/1625532 | address=0x47e06fec909ce3ed83c566eb3cfdbdd519069b26 | a_out=1 | a_in=0

Processing idx=45000/1625532 | address=0xa1cd4d12147091678e4fc749f903838810814d04 | a_

,idx,address,mode,seconds,a_out,a_in,cycle_3_value
129,1625511,0x21b92a8fe81f6f300c03d8a3a403aac32c2facc2,estimated,42.711722,15273,15273,6.228800e+10
139,1625521,0xcf076007e6c36cfb1eb2cd36ad66405fec7d0b31,estimated,8.806213,22129,4999,2.023956e+10
0,1625334,0x40a9f78879595e961fda688c69537c3529777426,exact,91.035107,2107,2131,1.774752e+09
49,1625431,0x0067cc2416f792cf0ec5629f5506324ac508f859,estimated,0.002710,4331,2489,1.020297e+05
33,1625415,0x391e7c679d29bd940d63be94ad22a25d25b5a604,estimated,0.091200,3669,3203,8.321292e+03
133,1625515,0x5babe600b9fcd5fb7b66c0611bf4896d967b23a1,estimated,0.004213,16193,2173,7.384008e+03
61,1625443,0x555ce236c0220695b68341bc48c68d52210cc35b,estimated,0.001698,4663,323,6.490896e+03
79,1625461,0x307576dd4f73f91bb8c4a2edb762938e8e067d31,estimated,0.001863,5657,20621,3.439456e+03
118,1625500,0xb23360ccdd9ed1b15d45e5d3824bb409c8d7c460,estimated,0.002350,10867,54,7.606900e+02
54,1625436,0xdfaa75323fb721e5f29d43859390f62cc4b600b8,estimated,0.002032,4447,7325,5.336400e+02


Saved diagnostics: /content/drive/MyDrive/Graph_Mining_Project/ethereum_gnn/cycle_3_estimation_diagnostics.csv


In [ ]:


# ---------------------------------------------------------
# 12. Convert to dataframe
# ---------------------------------------------------------

print("\nCreating dataframe...")

rows = []

for node, counts in motif_counts.items():

    row = {
        "address": node
    }

    row.update(counts)

    rows.append(row)

motif_df = pd.DataFrame(rows)

# ---------------------------------------------------------
# 13. Log-transform motif counts
# ---------------------------------------------------------
# Remove unused motif column
motif_df = motif_df.drop(
    columns=["temporal_fan_in_out"],
    errors="ignore"
)

motif_cols = [
    "temporal_out_star",
    "temporal_in_star",
    "temporal_chain",
    "temporal_reciprocal",
    "temporal_cycle_3"
]

for col in motif_cols:
    motif_df[col] = np.log1p(motif_df[col])

# ---------------------------------------------------------
# 14. Save
# ---------------------------------------------------------

motif_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\nSaved motif features:")
print(OUTPUT_PATH)

print("\nShape:")
print(motif_df.shape)

display(motif_df.head())


Creating dataframe...

Saved motif features:
/content/drive/MyDrive/Graph_Mining_Project/ethereum_gnn/gnn_temporal_motif_features_1h_total_ALL_NODES.csv

Shape:
(1828910, 6)


,address,temporal_out_star,temporal_in_star,temporal_chain,temporal_reciprocal,temporal_cycle_3
0,0x06354ce08f96bc961b9f5110d513e1bedf256ce6,6.342121,6.408529,6.682109,0.000000,0.000000
1,0xa1abfa21f80ecf401bd41365adbb6fef6fefdf09,14.613776,13.950441,14.247688,3.555348,4.427215
2,0x56eddb7aa87536c09ccc2793473599fd21a8b17f,14.772671,1.386294,7.408531,0.000000,0.000000
3,0xab97925eb84fe0260779f58b7cb08d77dcb1ee2b,13.885771,0.000000,6.066108,0.000000,0.000000
4,0x35e2ef3afb4a87607351bac1ca16c1510ba94398,2.772589,0.000000,1.386294,0.000000,0.000000


# new graph with motifs

In [ ]:

OUTPUT_DIR = "/content/drive/MyDrive/Graph_Mining_Project/ethereum_gnn/"

GRAPH_PATH = os.path.join(OUTPUT_DIR, "ethereum_full_graph.pt")
NODE_FEATURES_PATH = os.path.join(OUTPUT_DIR, "node_features.csv")
NODE_MAPPING_PATH = os.path.join(OUTPUT_DIR, "node_mapping.csv")
MOTIF_PATH = os.path.join(OUTPUT_DIR, "gnn_temporal_motif_features_1h_total_ALL_NODES.csv")

NEW_FEATURES_PATH = os.path.join(OUTPUT_DIR, "node_features_with_temporal_motifs.csv")
NEW_GRAPH_PATH = os.path.join(OUTPUT_DIR, "ethereum_full_graph_with_temporal_motifs.pt")

In [ ]:
# ---------------------------------------------------------
# 1. Load files
# ---------------------------------------------------------

data = torch.load(
    GRAPH_PATH,
    map_location="cpu",
    weights_only=False
)

node_features = pd.read_csv(NODE_FEATURES_PATH)
node_mapping = pd.read_csv(NODE_MAPPING_PATH)
motifs = pd.read_csv(MOTIF_PATH)

# ---------------------------------------------------------
# 2. Standardize identifiers
# ---------------------------------------------------------

if "node_id" not in node_features.columns:
    node_features["node_id"] = node_features.index

node_features["node_id"] = node_features["node_id"].astype(int)

node_mapping["node_id"] = node_mapping["node_id"].astype(int)
node_mapping["address"] = (
    node_mapping["address"]
    .astype(str)
    .str.lower()
    .str.strip()
)

motifs["address"] = (
    motifs["address"]
    .astype(str)
    .str.lower()
    .str.strip()
)

# Attach node_id to motif table
motifs = motifs.merge(
    node_mapping[["address", "node_id"]],
    on="address",
    how="inner"
)

print("Motif rows matched to graph:", len(motifs))

Motif rows matched to graph: 1828910


In [ ]:
# ---------------------------------------------------------
# 3. Merge motif features into node features
# ---------------------------------------------------------

motif_cols = [
    "temporal_out_star",
    "temporal_in_star",
    "temporal_chain",
    "temporal_reciprocal",
    "temporal_cycle_3"
]

# Safety: keep only motif columns that exist
motif_cols = [col for col in motif_cols if col in motifs.columns]

node_features_with_motifs = node_features.merge(
    motifs[["node_id"] + motif_cols],
    on="node_id",
    how="left"
)

# Missing motif values mean no counted motifs for that node
for col in motif_cols:
    node_features_with_motifs[col] = (
        node_features_with_motifs[col]
        .fillna(0.0)
        .replace([np.inf, -np.inf], 0.0)
    )

print("Original feature shape:", node_features.shape)
print("New feature shape:", node_features_with_motifs.shape)
print("Added motif features:", motif_cols)

Original feature shape: (1828910, 21)
New feature shape: (1828910, 26)
Added motif features: ['temporal_out_star', 'temporal_in_star', 'temporal_chain', 'temporal_reciprocal', 'temporal_cycle_3']


In [ ]:
# ---------------------------------------------------------
# 4. Rebuild data.x
# ---------------------------------------------------------

exclude_cols = ["node_id", "address"]

feature_cols = [
    col for col in node_features_with_motifs.columns
    if col not in exclude_cols
]

x = torch.tensor(
    node_features_with_motifs[feature_cols].values,
    dtype=torch.float
)

data.x = x

print(data)
print("New data.x shape:", data.x.shape)
print("Number of feature columns:", len(feature_cols))

Data(x=[1828910, 25], edge_index=[2, 6092594], edge_attr=[6092594, 2], num_nodes=1828910)
New data.x shape: torch.Size([1828910, 25])
Number of feature columns: 25


In [ ]:
# ---------------------------------------------------------
# 5. Save updated graph and feature table
# ---------------------------------------------------------

node_features_with_motifs.to_csv(
    NEW_FEATURES_PATH,
    index=False
)

torch.save(
    data,
    NEW_GRAPH_PATH
)

print("Saved:")
print(NEW_FEATURES_PATH)
print(NEW_GRAPH_PATH)

Saved:
/content/drive/MyDrive/Graph_Mining_Project/ethereum_gnn/node_features_with_temporal_motifs.csv
/content/drive/MyDrive/Graph_Mining_Project/ethereum_gnn/ethereum_full_graph_with_temporal_motifs.pt


In [ ]:
# =========================================================
# GraphSAGE semi-supervised node classification
# with validation-selected fraud threshold AND MOTIF
# =========================================================

# ---------------------------------------------------------
# 1. Paths
# ---------------------------------------------------------

OUTPUT_DIR = "/content/drive/MyDrive/Graph_Mining_Project/ethereum_gnn/"
GRAPH_PATH = os.path.join(OUTPUT_DIR, "ethereum_full_graph_with_temporal_motifs.pt")
LABEL_PATH = os.path.join(OUTPUT_DIR, "gnn_labels_02.csv")
NODE_MAPPING_PATH = os.path.join(OUTPUT_DIR, "node_mapping.csv")

# ---------------------------------------------------------
# 2. Load graph and labels
# ---------------------------------------------------------

data = torch.load(
    GRAPH_PATH,
    map_location="cpu",
    weights_only=False
)

labels_df = pd.read_csv(LABEL_PATH)
node_mapping = pd.read_csv(NODE_MAPPING_PATH)

labels_df["node_id"] = labels_df["node_id"].astype(int)
labels_df["label"] = labels_df["label"].astype(int)

print(data)
print(labels_df["label"].value_counts())
print("Total labelled nodes:", len(labels_df))

# ---------------------------------------------------------
# 3. Create y vector
# ---------------------------------------------------------
# -1 = unlabeled
#  0 = normal
#  1 = fraud

num_nodes = data.num_nodes

y = torch.full(
    (num_nodes,),
    -1,
    dtype=torch.long
)

labelled_node_ids = labels_df["node_id"].values
label_values = labels_df["label"].values

y[labelled_node_ids] = torch.tensor(
    label_values,
    dtype=torch.long
)

data.y = y

print("Unlabelled nodes:", (data.y == -1).sum().item())
print("Normal labelled nodes:", (data.y == 0).sum().item())
print("Fraud labelled nodes:", (data.y == 1).sum().item())

# ---------------------------------------------------------
# 4. Train / validation / test split
# ---------------------------------------------------------

train_ids, temp_ids = train_test_split(
    labelled_node_ids,
    test_size=0.30,
    random_state=42,
    stratify=label_values
)

temp_labels = y[temp_ids].numpy()

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)

train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)

train_mask[train_ids] = True
val_mask[val_ids] = True
test_mask[test_ids] = True

data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

print("Train nodes:", data.train_mask.sum().item())
print("Validation nodes:", data.val_mask.sum().item())
print("Test nodes:", data.test_mask.sum().item())

# ---------------------------------------------------------
# 5. Define GraphSAGE model
# ---------------------------------------------------------

class GraphSAGE(torch.nn.Module):
    def __init__(
        self,
        in_channels,
        hidden_channels,
        out_channels,
        dropout=0.2
    ):
        super().__init__()

        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )
        x = self.conv2(x, edge_index)
        return x

# ---------------------------------------------------------
# 6. Device, model, optimizer
# ---------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

data = data.to(device)

model = GraphSAGE(
    in_channels=data.x.shape[1],
    hidden_channels=64,
    out_channels=2,
    dropout=0.3
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=5e-4
)


# ---------------------------------------------------------
# 7. Class weights for imbalance
# ---------------------------------------------------------

train_labels = data.y[data.train_mask]

num_normal = (train_labels == 0).sum().item()
num_fraud = (train_labels == 1).sum().item()

class_weights = torch.tensor(
    [
        1.0 / num_normal,
        1.0 / num_fraud
    ],
    dtype=torch.float,
    device=device
)

class_weights = class_weights / class_weights.sum() * 2

print("Class weights:", class_weights)
# ---------------------------------------------------------
# 8. Training and evaluation functions
# ---------------------------------------------------------

def train_one_epoch():
    model.train()
    optimizer.zero_grad()

    out = model(data.x, data.edge_index)

    loss = F.cross_entropy(
        out[data.train_mask],
        data.y[data.train_mask],
        weight=class_weights
    )

    loss.backward()
    optimizer.step()

    return loss.item()


@torch.no_grad()
def evaluate(mask):
    model.eval()

    out = model(data.x, data.edge_index)

    logits = out[mask]
    labels = data.y[mask]

    probs = F.softmax(logits, dim=1)[:, 1]
    preds_argmax = logits.argmax(dim=1)

    acc = (preds_argmax == labels).float().mean().item()

    probs_np = probs.detach().cpu().numpy()
    preds_np = preds_argmax.detach().cpu().numpy()
    labels_np = labels.detach().cpu().numpy()

    roc_auc = roc_auc_score(labels_np, probs_np)
    pr_auc = average_precision_score(labels_np, probs_np)

    return acc, roc_auc, pr_auc, preds_np, probs_np, labels_np

# ---------------------------------------------------------
# 9. Train model
# ---------------------------------------------------------

EPOCHS = 150

best_val_pr_auc = 0
best_model_path = os.path.join(OUTPUT_DIR, "best_graphsage_model_03.pt")

for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch()

    if epoch % 10 == 0:
        train_acc, train_roc, train_pr, _, _, _ = evaluate(data.train_mask)
        val_acc, val_roc, val_pr, _, _, _ = evaluate(data.val_mask)

        print(
            f"Epoch {epoch:03d} | "
            f"Loss: {loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Train PR-AUC: {train_pr:.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"Val ROC-AUC: {val_roc:.4f} | "
            f"Val PR-AUC: {val_pr:.4f}"
        )

        if val_pr > best_val_pr_auc:
            best_val_pr_auc = val_pr

            torch.save(
                model.state_dict(),
                best_model_path
            )

            print("Saved best model.")

# ---------------------------------------------------------
# 10. Load best model
# ---------------------------------------------------------

model.load_state_dict(
    torch.load(
        best_model_path,
        map_location=device
    )
)

# ---------------------------------------------------------
# 11. Select best fraud threshold on validation set
# ---------------------------------------------------------
# Choose the threshold that maximizes F1-score on validation data.

val_acc, val_roc, val_pr, val_preds_argmax, val_probs, val_labels = evaluate(
    data.val_mask
)

thresholds = np.arange(0.05, 0.96, 0.01)

threshold_results = []

for threshold in thresholds:
    val_preds_thresholded = (val_probs >= threshold).astype(int)

    precision = precision_score(
        val_labels,
        val_preds_thresholded,
        zero_division=0
    )

    recall = recall_score(
        val_labels,
        val_preds_thresholded,
        zero_division=0
    )

    f1 = f1_score(
        val_labels,
        val_preds_thresholded,
        zero_division=0
    )

    threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_results = pd.DataFrame(threshold_results)

best_row = threshold_results.loc[
    threshold_results["f1"].idxmax()
]

best_threshold = float(best_row["threshold"])

print("\nBest threshold selected on validation set:")
print(best_row)

# Save threshold search results
threshold_results_path = os.path.join(
    OUTPUT_DIR,
    "graphsage_threshold_results_validation_03.csv"
)

threshold_results.to_csv(
    threshold_results_path,
    index=False
)

print("Saved threshold results:", threshold_results_path)

# ---------------------------------------------------------
# 12. Final test evaluation with selected threshold
# ---------------------------------------------------------

test_acc_argmax, test_roc, test_pr, test_preds_argmax, test_probs, test_labels = evaluate(
    data.test_mask
)

test_preds_thresholded = (test_probs >= best_threshold).astype(int)

print("\nFinal test results with validation-selected threshold")
print("Selected threshold:", best_threshold)
print("Test ROC-AUC:", test_roc)
print("Test PR-AUC:", test_pr)

print("\nClassification report:")
print(
    classification_report(
        test_labels,
        test_preds_thresholded,
        target_names=["normal", "fraud"],
        zero_division=0
    )
)

print("\nConfusion matrix:")
print(confusion_matrix(test_labels, test_preds_thresholded))

# ---------------------------------------------------------
# 13. Predict fraud probability for every node
# ---------------------------------------------------------

model.eval()

with torch.no_grad():
    out = model(data.x, data.edge_index)
    probs = F.softmax(out, dim=1)
    fraud_probs = probs[:, 1].detach().cpu().numpy()

pred_df = pd.DataFrame({
    "node_id": np.arange(data.num_nodes),
    "fraud_probability": fraud_probs
})

node_mapping["node_id"] = node_mapping["node_id"].astype(int)

pred_df = pred_df.merge(
    node_mapping,
    on="node_id",
    how="left"
)

# Mark whether the node was labelled or unlabeled
y_cpu = data.y.detach().cpu().numpy()

pred_df["known_label"] = y_cpu
pred_df["is_labelled"] = pred_df["known_label"] != -1

# Add thresholded prediction
pred_df["predicted_label_thresholded"] = (
    pred_df["fraud_probability"] >= best_threshold
).astype(int)

pred_df["predicted_label_name"] = pred_df[
    "predicted_label_thresholded"
].map({
    0: "normal",
    1: "fraud"
})

pred_df = pred_df.sort_values(
    "fraud_probability",
    ascending=False
)

PRED_OUTPUT_PATH = os.path.join(
    OUTPUT_DIR,
    "graphsage_all_node_predictions_New_labels.csv"
)

pred_df.to_csv(
    PRED_OUTPUT_PATH,
    index=False
)

print("\nSaved predictions:")
print(PRED_OUTPUT_PATH)

display(pred_df.head(20))

Data(x=[1828910, 25], edge_index=[2, 6092594], edge_attr=[6092594, 2], num_nodes=1828910)
label
0    45000
1     4466
Name: count, dtype: int64
Total labelled nodes: 49466
Unlabelled nodes: 1779444
Normal labelled nodes: 45000
Fraud labelled nodes: 4466
Train nodes: 34626
Validation nodes: 7420
Test nodes: 7420
Using device: cuda
Class weights: tensor([0.1806, 1.8194], device='cuda:0')
Epoch 010 | Loss: 0.7209 | Train Acc: 0.2904 | Train PR-AUC: 0.4300 | Val Acc: 0.2845 | Val ROC-AUC: 0.7850 | Val PR-AUC: 0.4292
Saved best model.
Epoch 020 | Loss: 0.5715 | Train Acc: 0.6549 | Train PR-AUC: 0.6151 | Val Acc: 0.6546 | Val ROC-AUC: 0.8932 | Val PR-AUC: 0.5987
Saved best model.
Epoch 030 | Loss: 0.4649 | Train Acc: 0.8463 | Train PR-AUC: 0.6634 | Val Acc: 0.8400 | Val ROC-AUC: 0.9177 | Val PR-AUC: 0.6432
Saved best model.
Epoch 040 | Loss: 0.4296 | Train Acc: 0.8593 | Train PR-AUC: 0.6935 | Val Acc: 0.8554 | Val ROC-AUC: 0.9405 | Val PR-AUC: 0.6741
Saved best model.
Epoch 050 | Loss: 0.361

,node_id,fraud_probability,address,known_label,is_labelled,predicted_label_thresholded,predicted_label_name
1806,1806,0.999994,0xa26148ae51fa8e787df319c04137602cc018b521,1,True,1,fraud
980,980,0.999993,0x974caa59e49682cda0ad2bbe82983419a2ecc400,1,True,1,fraud
46,46,0.999989,0x1ab4973a48dc892cd9971ece8e01dcc7688f8f23,1,True,1,fraud
80,80,0.999988,0xa1abfa21f80ecf401bd41365adbb6fef6fefdf09,1,True,1,fraud
27,27,0.999981,0x28c6c06298d514db089934071355e5743bf21d60,1,True,1,fraud
5605,5605,0.999976,0xb5d85cbf7cb3ee0d56b3bb207d5fc4b82f43f511,1,True,1,fraud
18224,18224,0.999971,0xeb943c230218e0da7b2ca18b81f7eb7fbbbe9665,1,True,1,fraud
512,512,0.999967,0xf30ba13e4b04ce5dc4d254ae5fa95477800f0eb0,-1,False,1,fraud
1090,1090,0.999958,0xa9ac43f5b5e38155a288d1a01d2cbc4478e14573,-1,False,1,fraud
403,403,0.999949,0x2cff890f0378a11913b6129b2e97417a2c302680,-1,False,1,fraud
